# 🍓 FRUTINOVELAS · ComfyUI en Colab (H3 + imágenes Qwen)

Esta libreta prepara **ComfyUI v0.40.0** con tu flujo **H3 EQUILIBRADO 9:16 (inicio / inicio+final)** y lo publica
de forma segura para que **FRUTINOVELAS Studio** (que corre en tu PC) lo use como motor de generación.

**Qué hace, en orden**

| Celda | Acción |
|---|---|
| 1 | Configuración (perfil, Drive, túnel) |
| Archivos del kit | Escribe scripts, manifiesto, perfiles y tu flujo exacto |
| 2 | Diagnóstico de GPU, VRAM, RAM y disco |
| 3 | Instala ComfyUI fijado al commit `3541246` + dependencias |
| 4–5 | Plan de modelos y descarga verificada (tamaño + SHA-256, revisiones fijadas) |
| 6 | Prepara los flujos (tu H3 + voces Ref2VA y plantillas Qwen opcionales) |
| 7 / 7B | Arranca ComfyUI, **exporta a formato API automáticamente** y valida contra el servidor real |
| 8 | Gateway con token + túnel Cloudflare |
| 9 / 9B | Datos de conexión para el programa (URL) y token (paso privado) |
| 10 | Pruebas reales opcionales (H3 5 s solo-inicio / inicio+final, Qwen) |
| 11 | Guardar en Drive y detener procesos |

**Antes de empezar**

1. *Entorno de ejecución → Cambiar tipo de entorno* → **GPU** (recomendado A100 80 GB, H100 o G4 con RAM alta).
2. Panel 🔑 **Secretos** (barra izquierda), todos opcionales:
   - `HF_TOKEN`: token de lectura de Hugging Face (evita límites de descarga).
   - `FRUTI_GATEWAY_TOKEN`: token fijo del gateway (≥ 16 caracteres). Si no existe se genera uno.
   - `CF_TUNNEL_TOKEN` y `CF_TUNNEL_HOSTNAME`: solo si usas un túnel con nombre de Cloudflare (URL estable).
3. Ejecuta las celdas **una a una, en orden**. Las celdas 10 y 11 tienen casillas desactivadas por defecto.

**Qwen-Image-2.1 BF16 viene activado** para imágenes 1152×2048 con 40 pasos.
Descarga 32,44 GB de pesos adicionales; usa su licencia Qwen Research.
Conserva los perfiles Qwen anteriores como opciones de regreso. No cambia el canvas H3.

**ACTIVAR_VOCES_PERSONAJES viene activado** para generar diálogo nuevo con una muestra guardada por personaje.
Añade el checkpoint Ref2VA INT8 (20,97 GB) y reutiliza el encoder y los VAEs H3 del perfil seleccionado.
La celda 4 muestra el espacio total antes de descargar. Puedes desactivar la opción si esta sesión no usará voces.
En Studio, guarda una muestra por personaje y selecciona sus voces en cada escena; no subas el diálogo completo.
Las muestras admiten 2–15 s cada una, hasta tres voces y 15 s en total por escena.
Actualiza Studio y su MCP a la misma versión del repositorio antes de conectar esta libreta.
El catálogo de estilos vive en Studio: el estilo angulado y geométrico solo se aplica al pedirlo,
y entonces la IA consulta referencias en internet mediante las herramientas de su cliente.

> ⚠️ **Avisos importantes**
> - Las [condiciones de Colab](https://research.google.com/colaboratory/faq.html) restringen en el plan gratuito el uso principal mediante interfaces web remotas y los proxies remotos. Se recomienda **plan de pago con saldo de cómputo**. Esta libreta **no** usa autoclic ni trucos anti-inactividad: mantén la pestaña abierta.
> - Un *Quick Tunnel* de Cloudflare es para pruebas: la URL cambia en cada sesión.
> - Requisitos de VRAM/RAM de H3 **no verificados** por FRUTINOVELAS. Hasta que la celda 10 termine con éxito, la compatibilidad real es **pendiente de prueba**.
> - Nunca compartas el token ni guardes la libreta con la salida de la celda 9B visible.


## 1 · Configuración

In [ ]:
# @title 1 · Configuración
PERFIL_VIDEO = "h3_balanced_default"  # @param ["h3_balanced_default", "h3_quality_bf16_optional"]
ACTIVAR_VOCES_PERSONAJES = True  # @param {type:"boolean"}
ACTIVAR_QWEN_21_BF16 = True  # @param {type:"boolean"}
ACTIVAR_QWEN_2512_T2I = False  # @param {type:"boolean"}
QWEN_EDIT_2511 = "no"  # @param ["no", "fp8mixed", "bf16"]
USAR_DRIVE = True  # @param {type:"boolean"}
GUARDAR_MODELOS_EN_DRIVE = False  # @param {type:"boolean"}
VERIFICAR_SHA256 = True  # @param {type:"boolean"}
MODO_TUNEL = "quick"  # @param ["quick", "named", "ninguno"]

import json
import os
import signal
import sys
import time
from pathlib import Path

PLATAFORMA = "colab"
BASE_DIR = Path("/content")
KIT_DIR = BASE_DIR / "FRUTINOVELAS_kit"
COMFY_DIR = BASE_DIR / "ComfyUI"
DATA_DIR = BASE_DIR / "FRUTINOVELAS_data"
MODELS_DIR = BASE_DIR / "FRUTINOVELAS_models"
DRIVE_ROOT = Path("/content/drive/MyDrive/FRUTINOVELAS")

IN_COLAB = "google.colab" in sys.modules
if USAR_DRIVE and IN_COLAB:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
if GUARDAR_MODELOS_EN_DRIVE:
    if not (USAR_DRIVE and IN_COLAB):
        raise SystemExit("GUARDAR_MODELOS_EN_DRIVE necesita USAR_DRIVE activado.")
    MODELS_DIR = DRIVE_ROOT / "models"
    print("⚠️ Los modelos (≈40 GB o más) se guardarán en tu Drive. Comprueba que tienes espacio.")
# Carpeta que sobrevive al cierre de la sesión (token, conexión, copias de cada sesión).
PERSIST_DIR = DRIVE_ROOT if (USAR_DRIVE and IN_COLAB) else None
# El entorno de Colab se borra al desconectar: la marca de dependencias vive con él.
PIP_MARKER = COMFY_DIR / ".frutinovelas_pip_ok"

NOTEBOOK_VERSION = "1.3.0"
COMFY_REPO = "https://github.com/Comfy-Org/ComfyUI.git"
COMFY_TAG = "v0.40.0"
COMFY_COMMIT = "3541246a9c997f4b8c32f9f88f3c985d02021c79"
TEMPLATES_COMMIT = "3e89967349dd95fa68ce3b36966f41d642af8ea7"
CLOUDFLARED_VERSION = "2026.10.0"
COMFY_PORT = 8188
GATEWAY_PORT = 8189
UA = f"FRUTINOVELAS-{PLATAFORMA}/{NOTEBOOK_VERSION}"
H3_PROFILE_KEY = "h3_equilibrado_9x16"
API_DIR = DATA_DIR / "workflows" / "api"
LOG_DIR = DATA_DIR / "logs"
UI_PATH = KIT_DIR / "workflows" / "H3_EQUILIBRADO_9x16_INICIO_FINAL.ui.json"

for d in [KIT_DIR / "scripts", KIT_DIR / "config", KIT_DIR / "workflows" / "templates", KIT_DIR / "bin",
          KIT_DIR / "src" / "frutinovelas" / "services",
          DATA_DIR / "workflows", API_DIR, DATA_DIR / "secrets", DATA_DIR / "tests",
          DATA_DIR / "comfy_input", DATA_DIR / "comfy_output", LOG_DIR, MODELS_DIR] + ([PERSIST_DIR] if PERSIST_DIR else []):
    d.mkdir(parents=True, exist_ok=True)
os.chmod(DATA_DIR / "secrets", 0o700)


def secreto(nombre):
    """Lee un secreto de Colab sin mostrarlo. Devuelve None si no existe o no hay permiso."""
    try:
        from google.colab import userdata

        valor = userdata.get(nombre)
        return valor.strip() if valor else None
    except Exception:
        return (os.environ.get(nombre) or "").strip() or None


STATE_FILE = DATA_DIR / "estado.json"
PIDS_FILE = DATA_DIR / "pids.json"


def estado(**cambios):
    """Estado persistente de la sesión (sin secretos)."""
    datos = json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {}
    datos.update(cambios)
    STATE_FILE.write_text(json.dumps(datos, ensure_ascii=False, indent=2))
    return datos


# Los PID solo valen dentro del mismo contenedor: /tmp se borra al reiniciar el pod o el entorno.
SESSION_FILE = Path("/tmp/frutinovelas_session")
if not SESSION_FILE.exists():
    import uuid

    SESSION_FILE.write_text(uuid.uuid4().hex)
SESSION_ID = SESSION_FILE.read_text().strip()
PROCS = json.loads(PIDS_FILE.read_text()) if PIDS_FILE.exists() else {}
if PROCS.get("_session") != SESSION_ID:
    PROCS = {"_session": SESSION_ID}
    if STATE_FILE.exists():
        estado(public_url=None, gateway_listen=None)


def guardar_pids():
    PIDS_FILE.write_text(json.dumps(PROCS, indent=2))


def vivo(pid):
    try:
        os.kill(int(pid), 0)
        return True
    except (OSError, TypeError, ValueError):
        return False


def cargar_kit():
    """Carga manifiesto, perfil H3 y el validador (recargado para que la celda sea repetible)."""
    import importlib

    if str(KIT_DIR / "scripts") not in sys.path:
        sys.path.insert(0, str(KIT_DIR / "scripts"))
    if str(KIT_DIR / "src") not in sys.path:
        sys.path.insert(0, str(KIT_DIR / "src"))
    import validate_workflows

    importlib.reload(validate_workflows)
    # Si se reescribió el kit en esta sesión, usar el helper actualizado.
    voice_module = sys.modules.get("frutinovelas.services.voice_workflow")
    if voice_module is not None:
        importlib.reload(voice_module)
    manifest = json.loads((KIT_DIR / "config" / "models_manifest.lock.json").read_text())
    profile = validate_workflows.load_profile(KIT_DIR / "config" / "workflow_profiles.json", H3_PROFILE_KEY)
    return manifest, profile, validate_workflows


PERFILES_ACTIVOS = [PERFIL_VIDEO]
if ACTIVAR_VOCES_PERSONAJES:
    PERFILES_ACTIVOS.append("h3_voice_references")
if ACTIVAR_QWEN_21_BF16:
    PERFILES_ACTIVOS.append("qwen21_bf16")
if ACTIVAR_QWEN_2512_T2I:
    PERFILES_ACTIVOS.append("qwen2512_t2i")
if QWEN_EDIT_2511 != "no":
    PERFILES_ACTIVOS.append({"fp8mixed": "qwen_edit2511_fp8mixed", "bf16": "qwen_edit2511_bf16"}[QWEN_EDIT_2511])

estado(perfiles=PERFILES_ACTIVOS, modo_tunel=MODO_TUNEL, notebook_version=NOTEBOOK_VERSION)
print("Perfiles de modelos:", ", ".join(PERFILES_ACTIVOS))
print("Modelos en:", MODELS_DIR)
print("Plataforma:", PLATAFORMA)
print("Persistencia:", PERSIST_DIR or "ninguna (todo se borra al cerrar la sesión)")
print("Túnel:", MODO_TUNEL)
print("Secretos detectados:", ", ".join(n for n in ["HF_TOKEN", "FRUTI_GATEWAY_TOKEN", "CF_TUNNEL_TOKEN", "CF_TUNNEL_HOSTNAME"] if secreto(n)) or "ninguno")

### Archivos del kit
Scripts y configuración del repositorio FRUTINOVELAS Studio, escritos tal cual. No hace falta tocarlos.

In [ ]:
%%writefile /content/FRUTINOVELAS_kit/src/frutinovelas/services/qwen21_workflow.py
"""Qwen 2.1 BF16 scene graph, using native nodes in the pinned ComfyUI commit.

Based on Comfy-Org/workflow_templates at 59a3b48bf369c5a15f29a1308bae3e279fa15505.
Scene composition deliberately uses a custom canvas, not the first reference's size.
No prompt enhancer, Lightning, or external custom nodes are required.
"""

T2I_WORKFLOW = "QWEN_21_T2I.api.json"
EDIT_WORKFLOW = "QWEN_21_EDIT.api.json"
WORKFLOWS = (T2I_WORKFLOW, EDIT_WORKFLOW)
PROFILE = "qwen21_bf16"
WIDTH, HEIGHT = 1152, 2048


def build_graph():
    def node(kind, **inputs):
        return {"class_type": kind, "inputs": inputs}

    return {
        "model": node("UNETLoader", unet_name="qwen_image_2.1_bf16.safetensors", weight_dtype="default"),
        "clip": node(
            "CLIPLoader", clip_name="qwen3vl_8b_bf16.safetensors", type="qwen_image", device="default"
        ),
        "vae": node("VAELoader", vae_name="qwen_image_2.1_vae_bf16.safetensors"),
        "encode": node(
            "TextEncodeQwenImage21",
            clip=["clip", 0],
            vae=["vae", 0],
            prompt="A strawberry character in a sunlit market.",
            negative_prompt="",
            resolution=1024,
        ),
        "cache": node("QwenImage21Cache", model=["model", 0], device="auto", dtype="default"),
        "canvas": node("EmptyLatentImage", width=WIDTH, height=HEIGHT, batch_size=1),
        "sampler": node(
            "KSampler",
            model=["cache", 0],
            positive=["encode", 0],
            negative=["encode", 1],
            latent_image=["canvas", 0],
            seed=42,
            steps=40,
            cfg=1.0,
            sampler_name="euler",
            scheduler="simple",
            denoise=1.0,
        ),
        "decode": node("VAEDecode", samples=["sampler", 0], vae=["vae", 0]),
        "rgb": node("SplitImageWithAlpha", image=["decode", 0]),
        "save": node("SaveImage", images=["rgb", 0], filename_prefix="FRUTINOVELAS/Qwen21"),
    }

In [ ]:
%%writefile /content/FRUTINOVELAS_kit/scripts/validate_workflows.py
"""Validación y utilidades de grafos ComfyUI para FRUTINOVELAS.

Sin dependencias externas: lo usan la libreta Colab, las pruebas y el programa.

- Detecta si un JSON es de interfaz (``nodes``/``links``) o de API.
- Valida un grafo API contra ``/object_info`` real: clases, entradas obligatorias,
  enlaces, opciones de combos (incluidos los nombres de modelos instalados) y rangos.
- Comprueba un perfil FRUTINOVELAS (IDs, clases y valores esperados).
- Deriva la variante solo-inicio (I2VA) eliminando de verdad ``last_frame``.
- Materializa un trabajo sobre una COPIA del grafo cambiando solo campos mapeados.

Uso por línea de órdenes::

    python validate_workflows.py H3_USUARIO_INICIO_FINAL.api.json \
        --object-info object_info.json --profiles workflow_profiles.json --mode fl2va
"""

from __future__ import annotations

import argparse
import copy
import json
import sys
from pathlib import Path
from typing import Any

# Entradas de LoadImage cuyo valor se sustituye en cada trabajo: no deben exigir
# que el placeholder exista en el servidor.
PLACEHOLDER_INPUTS = {("LoadImage", "image")}


# --------------------------------------------------------------------------- formato
def detect_format(data: Any) -> str:
    """Devuelve ``"ui"``, ``"api"`` o ``"unknown"``."""
    if isinstance(data, dict) and isinstance(data.get("nodes"), list) and "links" in data:
        return "ui"
    if isinstance(data, dict) and data and all(
        isinstance(v, dict) and "class_type" in v and isinstance(v.get("inputs"), dict) for v in data.values()
    ):
        return "api"
    return "unknown"


def is_link(value: Any) -> bool:
    return (
        isinstance(value, list)
        and len(value) == 2
        and isinstance(value[0], (str, int))
        and isinstance(value[1], int)
        and not isinstance(value[1], bool)
    )


def combo_options(spec: Any) -> list | None:
    """Opciones de un input combo en cualquiera de los formatos de /object_info."""
    if not isinstance(spec, (list, tuple)) or not spec:
        return None
    first = spec[0]
    if isinstance(first, list):
        return first
    if first == "COMBO" and len(spec) > 1 and isinstance(spec[1], dict):
        opts = spec[1].get("options")
        return list(opts) if isinstance(opts, (list, tuple)) else None
    return None


def _input_specs(info: dict) -> tuple[dict, dict]:
    inp = info.get("input") or {}
    return dict(inp.get("required") or {}), dict(inp.get("optional") or {})


# --------------------------------------------------------------------------- validación
def validate_api_graph(graph: dict, object_info: dict | None = None) -> dict:
    """Valida un grafo API. Devuelve ``{"errors": [...], "warnings": [...]}``."""
    errors: list[str] = []
    warnings: list[str] = []

    if detect_format(graph) != "api":
        return {"errors": ["El JSON no tiene formato API (se esperaba {id: {class_type, inputs}})."], "warnings": []}

    for node_id, node in graph.items():
        ctype = node["class_type"]
        inputs = node["inputs"]
        info = (object_info or {}).get(ctype) if object_info is not None else None
        if object_info is not None and info is None:
            errors.append(f"Nodo {node_id}: la clase '{ctype}' no existe en este ComfyUI (nodo ausente).")
            continue

        for name, value in inputs.items():
            if is_link(value):
                src_id = str(value[0])
                src = graph.get(src_id)
                if src is None:
                    errors.append(f"Nodo {node_id}.{name}: enlace a nodo inexistente {src_id}.")
                    continue
                if object_info is not None:
                    src_info = object_info.get(src["class_type"])
                    if src_info is not None:
                        n_out = len(src_info.get("output") or [])
                        if value[1] < 0 or value[1] >= n_out:
                            errors.append(
                                f"Nodo {node_id}.{name}: salida {value[1]} fuera de rango en {src_id} ({src['class_type']})."
                            )

        if info is None:
            continue
        required, optional = _input_specs(info)
        for name in required:
            if name not in inputs:
                # Native Qwen 2.1 declares an Autogrow group with min=0.
                # Text-to-image legitimately has no images.image_N inputs.
                if ctype == "TextEncodeQwenImage21" and name == "images":
                    continue
                # Los inputs dinámicos (Autogrow/DynamicCombo) se serializan con
                # claves con punto; si existe alguna con ese prefijo, se acepta.
                if any(k.startswith(name + ".") for k in inputs):
                    continue
                errors.append(f"Nodo {node_id} ({ctype}): falta la entrada obligatoria '{name}'.")
        all_specs = {**required, **optional}
        for name, value in inputs.items():
            if is_link(value) or name not in all_specs:
                continue
            spec = all_specs[name]
            opts = combo_options(spec)
            if opts is not None:
                if value not in opts:
                    msg = f"Nodo {node_id} ({ctype}).{name}: valor '{value}' no está entre las opciones del servidor."
                    if (ctype, name) in PLACEHOLDER_INPUTS:
                        warnings.append(msg + " (placeholder: el programa lo sustituye al subir la imagen)")
                    else:
                        errors.append(msg)
                continue
            if isinstance(spec, (list, tuple)) and spec and spec[0] in ("INT", "FLOAT"):
                cfg = spec[1] if len(spec) > 1 and isinstance(spec[1], dict) else {}
                if isinstance(value, (int, float)) and not isinstance(value, bool):
                    lo, hi = cfg.get("min"), cfg.get("max")
                    if lo is not None and value < lo:
                        errors.append(f"Nodo {node_id} ({ctype}).{name}: {value} < mínimo {lo}.")
                    if hi is not None and value > hi:
                        errors.append(f"Nodo {node_id} ({ctype}).{name}: {value} > máximo {hi}.")
                else:
                    errors.append(f"Nodo {node_id} ({ctype}).{name}: se esperaba número y llegó {value!r}.")

    outputs = [nid for nid, n in graph.items() if n["class_type"] in ("SaveVideo", "SaveImage", "SaveAnimatedWEBP", "SaveWEBM")]
    if not outputs:
        warnings.append("No se encontró ningún nodo de guardado conocido (SaveVideo/SaveImage).")
    return {"errors": errors, "warnings": warnings}


# --------------------------------------------------------------------------- perfiles
def load_profile(profiles_path: str | Path, key: str) -> dict:
    data = json.loads(Path(profiles_path).read_text(encoding="utf-8"))
    for p in data.get("profiles", []):
        if p.get("key") == key:
            return p
    raise KeyError(f"Perfil '{key}' no encontrado en {profiles_path}")


def _iter_mapping_refs(mapping: dict):
    for field, ref in mapping.items():
        if ref is None:
            continue
        refs = ref if isinstance(ref, list) else [ref]
        for r in refs:
            if isinstance(r, dict) and "node_id" in r:
                yield field, r


def check_profile(graph: dict, profile: dict, mode: str) -> dict:
    """Comprueba que el grafo API corresponde al perfil y al modo (fl2va/i2va)."""
    errors: list[str] = []
    warnings: list[str] = []
    mapping = profile["mapping"]
    disabled_nodes: set[str] = set()
    if mode == "i2va":
        dis = profile["modes"]["i2va"]["end_image_disable"]
        disabled_nodes = {str(n) for n in dis.get("remove_nodes", [])}

    for field, ref in _iter_mapping_refs(mapping):
        nid = str(ref["node_id"])
        if nid in disabled_nodes:
            if nid in graph:
                errors.append(f"Modo {mode}: el nodo {nid} ({field}) debería haberse eliminado.")
            continue
        node = graph.get(nid)
        if node is None:
            errors.append(f"Mapeo '{field}': el nodo {nid} no existe en el grafo API.")
            continue
        want = ref.get("class_type")
        if want and node["class_type"] != want:
            errors.append(f"Mapeo '{field}': el nodo {nid} es '{node['class_type']}', se esperaba '{want}'.")
        if field != "outputs" and ref.get("input") and ref["input"] not in node["inputs"]:
            errors.append(f"Mapeo '{field}': el nodo {nid} no tiene la entrada '{ref['input']}'.")

    for nid, expected in (profile.get("expected_values") or {}).items():
        if nid in disabled_nodes:
            continue
        node = graph.get(nid)
        if node is None:
            errors.append(f"Valor esperado: falta el nodo {nid}.")
            continue
        for k, v in expected.items():
            if k == "class_type":
                if node["class_type"] != v:
                    errors.append(f"Nodo {nid}: clase '{node['class_type']}' ≠ esperada '{v}'.")
                continue
            got = node["inputs"].get(k)
            if is_link(got):
                warnings.append(f"Nodo {nid}.{k}: está enlazado; no se compara con {v!r}.")
            elif got != v and not (isinstance(got, (int, float)) and isinstance(v, (int, float)) and float(got) == float(v)):
                errors.append(f"Nodo {nid}.{k}: {got!r} ≠ esperado {v!r}.")

    start = mapping.get("start_image")
    h3_id = str(mapping["positive_prompt"]["node_id"])
    h3 = graph.get(h3_id)
    if h3 is not None:
        ff = h3["inputs"].get("first_frame")
        if not (is_link(ff) and start and str(ff[0]) == str(start["node_id"])):
            errors.append(f"first_frame del nodo {h3_id} no está enlazado al LoadImage inicial.")
        lf = h3["inputs"].get("last_frame")
        if mode == "fl2va":
            end = mapping.get("end_image")
            if not (is_link(lf) and end and str(lf[0]) == str(end["node_id"])):
                errors.append(f"last_frame del nodo {h3_id} no está enlazado al LoadImage final.")
        elif mode == "i2va" and lf is not None:
            errors.append(f"Modo i2va: last_frame del nodo {h3_id} sigue presente.")
    return {"errors": errors, "warnings": warnings}


def derive_i2va(graph: dict, profile: dict) -> dict:
    """Copia del grafo sin imagen final: elimina el input y los nodos indicados.

    No duplica la imagen inicial ni inyecta una imagen vacía.
    """
    g = copy.deepcopy(graph)
    dis = profile["modes"]["i2va"]["end_image_disable"]
    if dis.get("strategy") != "remove_link":
        raise ValueError(f"Estrategia no soportada: {dis.get('strategy')}")
    consumer = g.get(str(dis["consumer"]["node_id"]))
    if consumer is None:
        raise ValueError("El nodo consumidor de la imagen final no existe en el grafo.")
    consumer["inputs"].pop(dis["consumer"]["input"], None)
    for nid in dis.get("remove_nodes", []):
        nid = str(nid)
        still_used = [
            other_id
            for other_id, other in g.items()
            if other_id != nid and any(is_link(v) and str(v[0]) == nid for v in other["inputs"].values())
        ]
        if still_used:
            raise ValueError(f"No se puede eliminar el nodo {nid}: lo usan también {still_used}.")
        g.pop(nid, None)
    return g


def apply_loader_overrides(graph: dict, overrides: dict[str, str]) -> tuple[dict, list[str]]:
    """Sustituye nombres de pesos exactos (perfil alternativo explícito)."""
    g = copy.deepcopy(graph)
    changes: list[str] = []
    if not overrides:
        return g, changes
    for nid, node in g.items():
        for k, v in node["inputs"].items():
            if isinstance(v, str) and v in overrides:
                node["inputs"][k] = overrides[v]
                changes.append(f"Nodo {nid} ({node['class_type']}).{k}: {v} → {overrides[v]}")
    return g, changes


# --------------------------------------------------------------------------- duración
def quantize_frames(seconds: float, rule: dict) -> tuple[int, float]:
    """Réplica de la fórmula del nodo de duración: frames efectivos y segundos reales."""
    fps = int(rule["fps"])
    step = int(rule["grid_step"])
    offset = int(rule["grid_offset"])
    min_frames = int(rule.get("min_frames", offset))
    base = max(min_frames, round(float(seconds) * fps))  # round() de Python, igual que simpleeval
    frames = base + (offset - base % step) % step
    return frames, frames / fps


# --------------------------------------------------------------------------- materializar
def materialize(graph: dict, profile: dict, values: dict) -> dict:
    """Copia del grafo con los campos mapeados sustituidos. Nada más se toca.

    ``values`` admite: start_image, end_image, positive_prompt, seed, duration,
    width, height, steps, filename_prefix.
    """
    g = copy.deepcopy(graph)
    mapping = profile["mapping"]
    for field, value in values.items():
        if value is None:
            continue
        ref = mapping.get(field)
        if ref is None:
            raise KeyError(f"El perfil no mapea el campo '{field}'.")
        refs = ref if isinstance(ref, list) else [ref]
        for r in refs:
            node = g.get(str(r["node_id"]))
            if node is None:
                raise KeyError(f"Campo '{field}': el nodo {r['node_id']} no está en el grafo.")
            if r.get("class_type") and node["class_type"] != r["class_type"]:
                raise ValueError(f"Campo '{field}': clase inesperada {node['class_type']}.")
            if r["input"] not in node["inputs"]:
                raise KeyError(f"Campo '{field}': el nodo {r['node_id']} no tiene '{r['input']}'.")
            if is_link(node["inputs"][r["input"]]):
                raise ValueError(f"Campo '{field}': la entrada está enlazada y no se puede fijar.")
            node["inputs"][r["input"]] = value
    return g


def ui_nodes_missing_in_api(ui: dict, api: dict) -> list[str]:
    """Nodos de la interfaz que no aparecen en la API (las notas son normales)."""
    out = []
    for n in ui.get("nodes", []):
        if str(n["id"]) not in api:
            out.append(f"{n['id']} ({n.get('type')})")
    return out


# --------------------------------------------------------------------------- CLI
def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description="Valida grafos ComfyUI para FRUTINOVELAS")
    ap.add_argument("graph")
    ap.add_argument("--object-info")
    ap.add_argument("--profiles")
    ap.add_argument("--profile-key", default="h3_equilibrado_9x16")
    ap.add_argument("--mode", choices=["fl2va", "i2va"])
    args = ap.parse_args(argv)

    data = json.loads(Path(args.graph).read_text(encoding="utf-8"))
    fmt = detect_format(data)
    print(f"Formato: {fmt}")
    if fmt != "api":
        print("ERROR: se necesita la exportación API (Workflow → Export (API)).")
        return 2
    oi = json.loads(Path(args.object_info).read_text(encoding="utf-8")) if args.object_info else None
    rep = validate_api_graph(data, oi)
    if args.profiles and args.mode:
        prof = load_profile(args.profiles, args.profile_key)
        r2 = check_profile(data, prof, args.mode)
        rep["errors"] += r2["errors"]
        rep["warnings"] += r2["warnings"]
    for w in rep["warnings"]:
        print("AVISO:", w)
    for e in rep["errors"]:
        print("ERROR:", e)
    print("OK" if not rep["errors"] else f"{len(rep['errors'])} error(es)")
    return 0 if not rep["errors"] else 1


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /content/FRUTINOVELAS_kit/scripts/export_workflow_api.py
"""Exporta un flujo de interfaz ComfyUI a formato API usando el propio frontend.

Abre la interfaz del ComfyUI en marcha con Chromium sin ventana, carga el JSON de
interfaz y llama a ``app.graphToPrompt()``: es exactamente lo que hace el menú
*Workflow → Export (API)*. Así se resuelven subgrafos, widgets dinámicos y
entradas que la conversión manual no puede adivinar.

Uso::

    python export_workflow_api.py --comfy http://127.0.0.1:8188 \
        --ui flujo.ui.json --out flujo.api.json

Requiere ``playwright`` y Chromium (``python -m playwright install --with-deps chromium``).
"""

from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

JS_FIND_APP = """
() => {
  const a = window.app || (window.comfyAPI && window.comfyAPI.app && window.comfyAPI.app.app);
  return !!(a && a.graph && typeof a.graphToPrompt === 'function' && typeof a.loadGraphData === 'function');
}
"""

JS_EXPORT = """
async ({workflow, required}) => {
  const a = window.app || window.comfyAPI.app.app;
  const LG = window.LiteGraph || (window.comfyAPI && window.comfyAPI.litegraph && window.comfyAPI.litegraph.LiteGraph);
  const missing = [];
  if (LG && LG.registered_node_types) {
    for (const t of required) { if (!LG.registered_node_types[t]) missing.push(t); }
  }
  if (missing.length) { return {error: 'Nodos no registrados en el frontend: ' + missing.join(', ')}; }
  await a.loadGraphData(workflow, true, false);
  await new Promise(r => setTimeout(r, 1500));
  const p = await a.graphToPrompt();
  return {output: p.output};
}
"""


def export_api(comfy_url: str, ui_workflow: dict, timeout_s: float = 180.0) -> dict:
    from playwright.sync_api import sync_playwright

    required = sorted(
        {
            n["type"]
            for n in ui_workflow.get("nodes", [])
            if n.get("type") not in ("Note", "MarkdownNote", "Reroute", "PrimitiveNode")
            and "-" not in str(n.get("type"))  # los UUID de subgrafos se resuelven en el frontend
        }
    )
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True, args=["--no-sandbox"])
        try:
            page = browser.new_page()
            page.goto(comfy_url.rstrip("/") + "/", wait_until="domcontentloaded", timeout=int(timeout_s * 1000))
            page.wait_for_function(JS_FIND_APP, timeout=int(timeout_s * 1000))
            # Margen para que las extensiones registren todos los nodos.
            page.wait_for_timeout(3000)
            result = page.evaluate(JS_EXPORT, {"workflow": ui_workflow, "required": required})
        finally:
            browser.close()
    if not isinstance(result, dict) or result.get("error"):
        raise RuntimeError((result or {}).get("error") or "La exportación no devolvió resultado.")
    output = result.get("output")
    if not isinstance(output, dict) or not output:
        raise RuntimeError("graphToPrompt devolvió un grafo vacío.")
    return output


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description="Exporta un flujo de interfaz ComfyUI a formato API")
    ap.add_argument("--comfy", default="http://127.0.0.1:8188")
    ap.add_argument("--ui", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--timeout", type=float, default=180.0)
    args = ap.parse_args(argv)

    ui = json.loads(Path(args.ui).read_text(encoding="utf-8"))
    if not (isinstance(ui, dict) and isinstance(ui.get("nodes"), list)):
        print("ERROR: el archivo no es un JSON de interfaz ComfyUI (falta 'nodes').", file=sys.stderr)
        return 2
    api = export_api(args.comfy, ui, args.timeout)
    out = Path(args.out)
    tmp = out.with_suffix(out.suffix + ".partial")
    tmp.write_text(json.dumps(api, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(out)
    print(f"API exportada: {out} ({len(api)} nodos)")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /content/FRUTINOVELAS_kit/scripts/comfy_gateway.py
"""Gateway autenticado delante de ComfyUI para FRUTINOVELAS Studio.

ComfyUI escucha solo en 127.0.0.1 dentro del runtime. Este proxy:

- exige un token (``Authorization: Bearer <token>`` o ``X-Frutinovelas-Token``);
- reenvía solo las rutas nativas que usa el programa (lista cerrada), con y sin
  el prefijo ``/api``; no expone gestor de nodos, ficheros de usuario ni otras rutas;
- transmite por streaming subidas y descargas grandes (``/view``) y el WebSocket ``/ws``;
- publica ``/frutinovelas/identity`` y ``/frutinovelas/workflows/<nombre>.api.json``
  para que el programa verifique el servidor y descargue los flujos exportados;
- nunca escribe tokens ni cabeceras en el log.

Uso::

    FRUTI_GATEWAY_TOKEN_FILE=/ruta/token.txt python comfy_gateway.py \
        --listen 127.0.0.1 --port 8189 --upstream http://127.0.0.1:8188 \
        --workflows-dir $DATA_DIR/workflows/api \
        --identity-file $DATA_DIR/identity.json

``--listen 0.0.0.0`` solo cuando un proxy externo (p. ej. el de RunPod) debe alcanzarlo;
el token se exige igualmente en todas las rutas salvo ``/healthz``.
"""

from __future__ import annotations

import argparse
import asyncio
import hashlib
import hmac
import json
import logging
import os
import re
import sys
import time
from collections import defaultdict, deque
from pathlib import Path

# pyrefly: ignore [missing-import]
from aiohttp import ClientConnectionError, ClientError, ClientSession, ClientTimeout, WSMsgType, web

log = logging.getLogger("frutinovelas.gateway")

HOP_BY_HOP = {
    "connection",
    "keep-alive",
    "proxy-authenticate",
    "proxy-authorization",
    "te",
    "trailers",
    "transfer-encoding",
    "upgrade",
    "host",
    "authorization",
    "x-frutinovelas-token",
    "cookie",
}

GET_ROUTES = [
    r"/system_stats",
    r"/object_info",
    r"/object_info/[^/]+",
    r"/queue",
    r"/history",
    r"/history/[0-9a-fA-F-]{1,64}",
    r"/view",
    r"/prompt",
    r"/features",
    r"/models",
    r"/models/[A-Za-z0-9_\-]+",
    r"/embeddings",
]
POST_ROUTES = [r"/upload/image", r"/prompt", r"/queue", r"/interrupt"]
WORKFLOW_NAME = re.compile(r"^[A-Za-z0-9_.\-]+\.api\.json$")
MAX_UPLOAD_BYTES = 64 * 1024 * 1024
SESSION_KEY = web.AppKey("session", ClientSession)


def _compile(routes: list[str]) -> re.Pattern:
    return re.compile(r"^(?:/api)?(?:" + "|".join(routes) + r")$")


GET_RE = _compile(GET_ROUTES)
POST_RE = _compile(POST_ROUTES)
WS_RE = re.compile(r"^(?:/api)?/ws$")


class FailureLimiter:
    """Limita intentos fallidos de autenticación por IP."""

    def __init__(self, max_failures: int = 20, window_s: float = 60.0):
        self.max_failures = max_failures
        self.window_s = window_s
        self.events: dict[str, deque] = defaultdict(deque)

    def blocked(self, ip: str) -> bool:
        q = self.events[ip]
        now = time.monotonic()
        while q and now - q[0] > self.window_s:
            q.popleft()
        return len(q) >= self.max_failures

    def fail(self, ip: str) -> None:
        self.events[ip].append(time.monotonic())


def extract_token(request: web.Request) -> str | None:
    auth = request.headers.get("Authorization", "")
    if auth.lower().startswith("bearer "):
        return auth[7:].strip()
    tok = request.headers.get("X-Frutinovelas-Token")
    return tok.strip() if tok else None


def create_app(
    upstream: str,
    token: str,
    workflows_dir: Path | None = None,
    identity_file: Path | None = None,
) -> web.Application:
    if not token or len(token) < 16:
        raise ValueError("El token del gateway debe tener al menos 16 caracteres.")
    upstream = upstream.rstrip("/")
    limiter = FailureLimiter()

    @web.middleware
    async def auth_mw(request: web.Request, handler):
        if request.path == "/healthz":
            return await handler(request)
        ip = request.headers.get("CF-Connecting-IP") or request.remote or "?"
        tok = extract_token(request)
        if not tok or not hmac.compare_digest(tok.encode(), token.encode()):
            if limiter.blocked(ip):
                return web.json_response({"error": "demasiados intentos"}, status=429)
            limiter.fail(ip)
            return web.json_response({"error": "no autorizado"}, status=401)
        return await handler(request)

    app = web.Application(middlewares=[auth_mw], client_max_size=MAX_UPLOAD_BYTES)

    async def on_startup(app_: web.Application):
        app_[SESSION_KEY] = ClientSession(
            timeout=ClientTimeout(total=None, connect=15, sock_read=None),
            auto_decompress=False,
        )

    async def on_cleanup(app_: web.Application):
        await app_[SESSION_KEY].close()

    app.on_startup.append(on_startup)
    app.on_cleanup.append(on_cleanup)

    async def healthz(_request: web.Request):
        return web.json_response({"ok": True})

    async def identity(_request: web.Request):
        if identity_file and identity_file.is_file():
            data = read_identity()
            if data is None:
                return web.json_response({"error": "identidad no válida; ejecuta fruti start"}, status=503)
            return web.json_response(data)
        return web.json_response({"error": "identidad no disponible"}, status=404)

    def read_identity() -> dict | None:
        if not identity_file:
            return None
        try:
            data = json.loads(identity_file.read_text(encoding="utf-8"))
            if isinstance(data, dict) and isinstance(data.get("workflows"), dict):
                return data
        except (OSError, ValueError):
            pass
        return None

    def published_names() -> list[str]:
        if not workflows_dir or not workflows_dir.is_dir():
            return []
        data = read_identity()
        # Una identidad configurada es el inventario validado del arranque actual.
        # Los JSON de perfiles desactivados pueden seguir en disco como respaldo.
        if identity_file and data is None:
            return []
        return sorted(
            p.name
            for p in workflows_dir.iterdir()
            if WORKFLOW_NAME.fullmatch(p.name)
            and p.is_file()
            and p.resolve().parent == workflows_dir.resolve()
            and (data is None or p.name in data["workflows"])
        )

    async def list_workflows(_request: web.Request):
        return web.json_response({"workflows": published_names()})

    async def get_workflow(request: web.Request):
        name = request.match_info["name"]
        if not WORKFLOW_NAME.fullmatch(name) or not workflows_dir:
            return web.json_response({"error": "nombre no válido"}, status=400)
        path = (workflows_dir / name).resolve()
        if path.parent != workflows_dir.resolve() or name not in published_names():
            return web.json_response({"error": "no encontrado"}, status=404)
        body = path.read_bytes()
        data = read_identity()
        if data is not None and hashlib.sha256(body).hexdigest() != data["workflows"].get(name):
            return web.json_response(
                {"error": "workflow modificado; ejecuta fruti start --reexportar"}, status=409
            )
        return web.Response(body=body, content_type="application/json")

    async def proxy_ws(request: web.Request):
        session = request.app[SESSION_KEY]
        target = upstream.replace("http://", "ws://").replace("https://", "wss://") + request.path_qs
        client_ws = web.WebSocketResponse(heartbeat=30, max_msg_size=0)
        await client_ws.prepare(request)
        try:
            async with session.ws_connect(target, heartbeat=30, max_msg_size=0) as up_ws:

                async def up_to_client():
                    async for msg in up_ws:
                        if msg.type == WSMsgType.TEXT:
                            await client_ws.send_str(msg.data)
                        elif msg.type == WSMsgType.BINARY:
                            await client_ws.send_bytes(msg.data)
                        elif msg.type in (WSMsgType.CLOSE, WSMsgType.ERROR):
                            break

                async def client_to_up():
                    async for msg in client_ws:
                        if msg.type == WSMsgType.TEXT:
                            await up_ws.send_str(msg.data)
                        elif msg.type == WSMsgType.BINARY:
                            await up_ws.send_bytes(msg.data)
                        elif msg.type in (WSMsgType.CLOSE, WSMsgType.ERROR):
                            break

                tasks = [asyncio.create_task(up_to_client()), asyncio.create_task(client_to_up())]
                _done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
                for t in pending:
                    t.cancel()
                await asyncio.gather(*tasks, return_exceptions=True)
        except Exception as exc:  # noqa: BLE001
            log.warning("WS upstream error: %s", type(exc).__name__)
        finally:
            await client_ws.close()
        return client_ws

    async def proxy(request: web.Request):
        path = request.path
        method = request.method.upper()
        if (
            method == "GET"
            and WS_RE.match(path)
            and request.headers.get("Upgrade", "").lower() == "websocket"
        ):
            return await proxy_ws(request)
        if not ((method == "GET" and GET_RE.match(path)) or (method == "POST" and POST_RE.match(path))):
            return web.json_response({"error": "ruta no permitida por el gateway"}, status=403)
        clen = request.content_length
        if clen is not None and clen > MAX_UPLOAD_BYTES:
            return web.json_response({"error": "cuerpo demasiado grande"}, status=413)

        session = request.app[SESSION_KEY]
        headers = {k: v for k, v in request.headers.items() if k.lower() not in HOP_BY_HOP}
        data = request.content if request.can_read_body else None
        started = time.monotonic()
        resp = None
        try:
            async with session.request(
                method, upstream + request.path_qs, headers=headers, data=data, allow_redirects=False
            ) as up:
                resp_headers = {k: v for k, v in up.headers.items() if k.lower() not in HOP_BY_HOP}
                resp = web.StreamResponse(status=up.status, headers=resp_headers)
                await resp.prepare(request)
                async for chunk in up.content.iter_chunked(1 << 20):
                    await resp.write(chunk)
                await resp.write_eof()
        except (ClientError, TimeoutError) as exc:
            log.warning("ComfyUI upstream error: %s", type(exc).__name__)
            if resp is not None and resp.prepared:
                # No convertir una descarga incompleta en un archivo válido con HTTP 200.
                if request.transport:
                    request.transport.close()
                return resp
            status = (
                503
                if isinstance(exc, ClientConnectionError)
                else 504
                if isinstance(exc, TimeoutError)
                else 502
            )
            return web.json_response(
                {
                    "error": "ComfyUI no está disponible; comprueba que esté iniciado.",
                    "code": "comfyui_unavailable",
                    "gateway_available": True,
                },
                status=status,
            )
        log.info("%s %s -> %s (%.2fs)", method, path, up.status, time.monotonic() - started)
        return resp

    app.router.add_get("/healthz", healthz)
    app.router.add_get("/frutinovelas/identity", identity)
    app.router.add_get("/frutinovelas/workflows", list_workflows)
    app.router.add_get("/frutinovelas/workflows/{name}", get_workflow)
    app.router.add_route("*", "/{tail:.*}", proxy)
    return app


def _read_token() -> str:
    path = os.environ.get("FRUTI_GATEWAY_TOKEN_FILE")
    if path and Path(path).is_file():
        return Path(path).read_text(encoding="utf-8").strip()
    tok = os.environ.get("FRUTI_GATEWAY_TOKEN", "").strip()
    if tok:
        return tok
    raise SystemExit("Falta el token: define FRUTI_GATEWAY_TOKEN_FILE o FRUTI_GATEWAY_TOKEN.")


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description="Gateway autenticado para ComfyUI (FRUTINOVELAS)")
    ap.add_argument("--listen", default="127.0.0.1")
    ap.add_argument("--port", type=int, default=8189)
    ap.add_argument("--upstream", default="http://127.0.0.1:8188")
    ap.add_argument("--workflows-dir")
    ap.add_argument("--identity-file")
    args = ap.parse_args(argv)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", stream=sys.stderr)
    app = create_app(
        args.upstream,
        _read_token(),
        Path(args.workflows_dir) if args.workflows_dir else None,
        Path(args.identity_file) if args.identity_file else None,
    )
    web.run_app(app, host=args.listen, port=args.port, access_log=None, print=None)
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile /content/FRUTINOVELAS_kit/config/models_manifest.lock.json
{
  "schema_version": "1.0",
  "generated_at": "2026-10-14T23:10:00Z",
  "note": "Revisiones, tamaÃ±os y SHA-256 consultados en la API de Hugging Face el 2026-10-14. Ref2VA aÃ±adido el 2026-10-17, verificado contra el puntero LFS de la misma revisiÃ³n fijada. La libreta vuelve a comprobarlos antes de descargar. Qwen 2.1 BF16: revisión, tamaños y SHA-256 LFS verificados el 2026-10-18; no se descargaron pesos para esta actualización.",
  "repos": {
    "Comfy-Org/MiniMax-H3": "e5eb578a89295337b8ff433a035929ce0279e0b6",
    "Comfy-Org/Qwen-Image_ComfyUI": "1f12b17be14c89b026c51a91d67c32f84bb047bc",
    "Comfy-Org/Qwen-Image-Edit_ComfyUI": "f68ace85e60b4a02a323e394253731947657b7d2",
    "lightx2v/Qwen-Image-2512-Lightning": "a52649c9d0f6e1a248bff13f0df33bb8a2abdb52",
    "lightx2v/Qwen-Image-Edit-2511-Lightning": "d74eba145674fd7e31b949324e148e21e7118abd",
    "Comfy-Org/Qwen-Image-2.1": "a8342359c61979b06a8dafb1a47991fe1f30e994"
  },
  "components": {
    "h3_diffusion_fl2va_pruned_int8": {
      "repo_id": "Comfy-Org/MiniMax-H3",
      "path": "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors",
      "dest": "diffusion_models",
      "size": 20970379616,
      "sha256": "e889202c41dafb67b10d67b97f0d8541508036a6090af23425a5c2615d03c47a"
    },
    "h3_diffusion_ref2va_pruned_int8": {
      "repo_id": "Comfy-Org/MiniMax-H3",
      "path": "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
      "dest": "diffusion_models",
      "size": 20970379616,
      "sha256": "9255f52b6677845ad238f20dfaafa94727053694127ab7f255c048f0f9365779"
    },
    "h3_text_encoder_nvfp4_awq": {
      "repo_id": "Comfy-Org/MiniMax-H3",
      "path": "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
      "dest": "text_encoders",
      "size": 15687142551,
      "sha256": "35a88d51044231fe332301d7a62aa81e3f2cba62febeb446e2c1e3e0ef76f2c6"
    },
    "h3_video_vae_int8": {
      "repo_id": "Comfy-Org/MiniMax-H3",
      "path": "vae/minimax_h3_video_vae_int8_convrot.safetensors",
      "dest": "vae",
      "size": 2811065184,
      "sha256": "52a2c8c73583c86e4f41cdcce3a6ad0ea562987bc0bf3d60a0cef5f5c8e60c0e"
    },
    "h3_audio_vae_fp32": {
      "repo_id": "Comfy-Org/MiniMax-H3",
      "path": "vae/minimax_h3_audio_vae_fp32.safetensors",
      "dest": "vae",
      "size": 605254808,
      "sha256": "8e505d95dd1561d47abd43d4238fd40d9bb1ae9e147ed0a4cba778d76ae4db48"
    },
    "h3_diffusion_fl2va_pruned_bf16": {
      "repo_id": "Comfy-Org/MiniMax-H3",
      "path": "diffusion_models/minimax_h3_fl2va_pruned_bf16.safetensors",
      "dest": "diffusion_models",
      "size": 40225724176,
      "sha256": "a32572fb90b5508b201ec7c2eddcc184b13ddfd3c6f6d2cf06a0b46535d541b4"
    },
    "h3_text_encoder_bf16": {
      "repo_id": "Comfy-Org/MiniMax-H3",
      "path": "text_encoders/qwen3vl_32b_minimax_h3_bf16.safetensors",
      "dest": "text_encoders",
      "size": 51506295256,
      "sha256": "600d567f6a9629c8574e8e7041b199bdd9c59a986afa7906910a81919610607d"
    },
    "h3_video_vae_fp16": {
      "repo_id": "Comfy-Org/MiniMax-H3",
      "path": "vae/minimax_h3_video_vae_fp16.safetensors",
      "dest": "vae",
      "size": 5207808496,
      "sha256": "7c1f131492e7eddacaac9069a61b81bdd39de5cc96561e677c5eab1cdce5e522"
    },
    "qwen2512_diffusion_fp8": {
      "repo_id": "Comfy-Org/Qwen-Image_ComfyUI",
      "path": "split_files/diffusion_models/qwen_image_2512_fp8_e4m3fn.safetensors",
      "dest": "diffusion_models",
      "size": 20430679144,
      "sha256": "5dc80554d5d83390046a2f4a94ece06afb7700bf7b0aaf8bde9769793875876b"
    },
    "qwen_text_encoder_2_5_vl_7b_fp8": {
      "repo_id": "Comfy-Org/Qwen-Image_ComfyUI",
      "path": "split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors",
      "dest": "text_encoders",
      "size": 9384670680,
      "sha256": "cb5636d852a0ea6a9075ab1bef496c0db7aef13c02350571e388aea959c5c0b4"
    },
    "qwen_image_vae": {
      "repo_id": "Comfy-Org/Qwen-Image_ComfyUI",
      "path": "split_files/vae/qwen_image_vae.safetensors",
      "dest": "vae",
      "size": 253806246,
      "sha256": "a70580f0213e67967ee9c95f05bb400e8fb08307e017a924bf3441223e023d1f"
    },
    "qwen2512_lightning_lora": {
      "repo_id": "lightx2v/Qwen-Image-2512-Lightning",
      "path": "Qwen-Image-2512-Lightning-4steps-V1.0-fp32.safetensors",
      "dest": "loras",
      "size": 1698951104,
      "sha256": "ad12117461cb41e2ea637fec8df6392ce8e8550c47fbe2b829ed3deb98262066"
    },
    "qwen_edit2511_diffusion_fp8mixed": {
      "repo_id": "Comfy-Org/Qwen-Image-Edit_ComfyUI",
      "path": "split_files/diffusion_models/qwen_image_edit_2511_fp8mixed.safetensors",
      "dest": "diffusion_models",
      "size": 20533762817,
      "sha256": "c9fdc158e46d3b61ef75f21ae866ca2fe808bf4a53643120d1c1e87c19280a4e"
    },
    "qwen_edit2511_diffusion_bf16": {
      "repo_id": "Comfy-Org/Qwen-Image-Edit_ComfyUI",
      "path": "split_files/diffusion_models/qwen_image_edit_2511_bf16.safetensors",
      "dest": "diffusion_models",
      "size": 40861031560,
      "sha256": "ae42d927b5fac4f278b9a894554c727e619727a63622976f2d95625be4bce08c"
    },
    "qwen_edit2511_lightning_lora": {
      "repo_id": "lightx2v/Qwen-Image-Edit-2511-Lightning",
      "path": "Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors",
      "dest": "loras",
      "size": 849608296,
      "sha256": "22226e8d05d354bb356627d428809f5afd7819399b077238a2b70a82883a904f"
    },
    "qwen21_diffusion_bf16": {
      "repo_id": "Comfy-Org/Qwen-Image-2.1",
      "path": "diffusion_models/qwen_image_2.1_bf16.safetensors",
      "dest": "diffusion_models",
      "size": 14230280616,
      "sha256": "89f4158d066cc33906a199fca85634f766892dd78f49b6698dabf187ac86c4bc"
    },
    "qwen21_encoder_bf16": {
      "repo_id": "Comfy-Org/Qwen-Image-2.1",
      "path": "text_encoders/qwen3vl_8b_bf16.safetensors",
      "dest": "text_encoders",
      "size": 17534334616,
      "sha256": "68bdc82bc1b66851162ae656225e7e2068166b603db19bd5d5a3b90eb12669a9"
    },
    "qwen21_vae_bf16": {
      "repo_id": "Comfy-Org/Qwen-Image-2.1",
      "path": "vae/qwen_image_2.1_vae_bf16.safetensors",
      "dest": "vae",
      "size": 675509688,
      "sha256": "bb21f7473051e1ac368515dd3f2e15cd44d7a11748ee8823e1ddca3e4876b7c9"
    }
  },
  "profiles": {
    "h3_balanced_default": {
      "label": "H3 equilibrado (predeterminado): FL2VA INT8 + encoder NVFP4 AWQ + VAE INT8 + audio FP32",
      "components": [
        "h3_diffusion_fl2va_pruned_int8",
        "h3_text_encoder_nvfp4_awq",
        "h3_video_vae_int8",
        "h3_audio_vae_fp32"
      ],
      "loader_overrides": {}
    },
    "h3_quality_bf16_optional": {
      "label": "H3 alta precisiÃ³n (opcional): FL2VA BF16 + encoder BF16 + VAE FP16 + audio FP32",
      "components": [
        "h3_diffusion_fl2va_pruned_bf16",
        "h3_text_encoder_bf16",
        "h3_video_vae_fp16",
        "h3_audio_vae_fp32"
      ],
      "loader_overrides": {
        "minimax_h3_fl2va_pruned_int8_convrot.safetensors": "minimax_h3_fl2va_pruned_bf16.safetensors",
        "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": "qwen3vl_32b_minimax_h3_bf16.safetensors",
        "minimax_h3_video_vae_int8_convrot.safetensors": "minimax_h3_video_vae_fp16.safetensors"
      }
    },
    "h3_voice_references": {
      "label": "Voces de personajes (opcional): Ref2VA INT8; reutiliza encoder y VAEs del perfil H3 seleccionado",
      "components": [
        "h3_diffusion_ref2va_pruned_int8"
      ],
      "loader_overrides": {}
    },
    "qwen2512_t2i": {
      "label": "Qwen-Image-2512 (texto a imagen, FP8, plantilla oficial)",
      "components": [
        "qwen2512_diffusion_fp8",
        "qwen_text_encoder_2_5_vl_7b_fp8",
        "qwen_image_vae",
        "qwen2512_lightning_lora"
      ],
      "template": "templates/image_qwen_Image_2512.json",
      "loader_overrides": {}
    },
    "qwen_edit2511_fp8mixed": {
      "label": "Qwen-Image-Edit-2511 (ediciÃ³n, FP8 mixed, plantilla oficial)",
      "components": [
        "qwen_edit2511_diffusion_fp8mixed",
        "qwen_text_encoder_2_5_vl_7b_fp8",
        "qwen_image_vae",
        "qwen_edit2511_lightning_lora"
      ],
      "template": "templates/image_qwen_image_edit_2511.json",
      "loader_overrides": {}
    },
    "qwen_edit2511_bf16": {
      "label": "Qwen-Image-Edit-2511 (ediciÃ³n, difusiÃ³n BF16)",
      "components": [
        "qwen_edit2511_diffusion_bf16",
        "qwen_text_encoder_2_5_vl_7b_fp8",
        "qwen_image_vae",
        "qwen_edit2511_lightning_lora"
      ],
      "template": "templates/image_qwen_image_edit_2511.json",
      "loader_overrides": {
        "qwen_image_edit_2511_fp8mixed.safetensors": "qwen_image_edit_2511_bf16.safetensors"
      }
    },
    "qwen21_bf16": {
      "label": "Qwen-Image-2.1 BF16: generación y referencias, 1152×2048, 40 pasos",
      "components": [
        "qwen21_diffusion_bf16",
        "qwen21_encoder_bf16",
        "qwen21_vae_bf16"
      ],
      "loader_overrides": {},
      "workflow_builder": "frutinovelas.services.qwen21_workflow",
      "license": "qwen-research"
    }
  }
}

In [ ]:
%%writefile /content/FRUTINOVELAS_kit/config/workflow_profiles.json
{
  "schema_version": "1.0",
  "note": "Perfiles de flujo para FRUTINOVELAS Studio. Los node_id provienen del JSON del usuario y se comprueban contra la exportación API real (class_type incluido) antes de usarse.",
  "profiles": [
    {
      "key": "h3_equilibrado_9x16",
      "name": "H3 equilibrado 9:16 (inicio / inicio+final)",
      "kind": "video_generation",
      "engine": "MiniMax H3 FL2VA",
      "workflow_ui_file": "workflows/H3_EQUILIBRADO_9x16_INICIO_FINAL.ui.json",
      "workflow_ui_sha256": "2153a2f6d289aaeb38882c4eb364a06ef3b039ad93addc4347b0473340c9e547",
      "workflow_api_files": {
        "fl2va": "H3_USUARIO_INICIO_FINAL.api.json",
        "i2va": "H3_USUARIO_SOLO_INICIO.api.json"
      },
      "status": "pendiente_prueba_real",
      "mapping": {
        "start_image": {"node_id": "1", "input": "image", "class_type": "LoadImage"},
        "end_image": {"node_id": "30", "input": "image", "class_type": "LoadImage"},
        "positive_prompt": {"node_id": "9", "input": "prompt", "class_type": "MiniMaxH3ImageToVideo"},
        "negative_prompt": null,
        "seed": [{"node_id": "12", "input": "noise_seed", "class_type": "RandomNoise"}],
        "duration": {"node_id": "28", "input": "value", "class_type": "PrimitiveFloat", "unit": "seconds"},
        "width": {"node_id": "9", "input": "width", "class_type": "MiniMaxH3ImageToVideo"},
        "height": {"node_id": "9", "input": "height", "class_type": "MiniMaxH3ImageToVideo"},
        "steps": {"node_id": "14", "input": "steps", "class_type": "BasicScheduler"},
        "filename_prefix": {"node_id": "20", "input": "filename_prefix", "class_type": "SaveVideo"},
        "outputs": [{"node_id": "20", "class_type": "SaveVideo", "adapter": "comfy_save_video"}]
      },
      "modes": {
        "fl2va": {"requires": ["start_image", "end_image"]},
        "i2va": {
          "requires": ["start_image"],
          "end_image_disable": {
            "strategy": "remove_link",
            "consumer": {"node_id": "9", "input": "last_frame"},
            "remove_nodes": ["30"]
          }
        }
      },
      "duration_rule": {
        "fps": 24,
        "grid_step": 17,
        "grid_offset": 5,
        "min_frames": 5,
        "rounding": "round_then_snap_up",
        "formula": "max(5, round(s*24)) + (5 - max(5, round(s*24)) % 17) % 17",
        "published_min_seconds": 4,
        "published_max_seconds": 15
      },
      "image_handling": {
        "start_image": "El nodo H3 estira la imagen inicial al canvas sin recortar: prepararla en 9:16 exacto.",
        "end_image": "El nodo H3 recorta al centro la imagen final para cubrir el canvas."
      },
      "defaults": {"width": 576, "height": 1024, "steps": 20, "fps": 24, "duration_seconds": 10, "seed_mode": "random"},
      "locked": ["fps", "sampler_name", "scheduler", "denoise", "unet_name", "clip_name", "vae_name"],
      "expected_values": {
        "1": {"class_type": "LoadImage"},
        "5": {"class_type": "UNETLoader", "unet_name": "minimax_h3_fl2va_pruned_int8_convrot.safetensors"},
        "6": {"class_type": "CLIPLoader", "clip_name": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"},
        "7": {"class_type": "VAELoader", "vae_name": "minimax_h3_video_vae_int8_convrot.safetensors"},
        "8": {"class_type": "VAELoader", "vae_name": "minimax_h3_audio_vae_fp32.safetensors"},
        "9": {"class_type": "MiniMaxH3ImageToVideo", "width": 576, "height": 1024},
        "12": {"class_type": "RandomNoise"},
        "13": {"class_type": "KSamplerSelect", "sampler_name": "res_multistep"},
        "14": {"class_type": "BasicScheduler", "scheduler": "simple", "steps": 20, "denoise": 1},
        "19": {"class_type": "CreateVideo", "fps": 24},
        "20": {"class_type": "SaveVideo"},
        "28": {"class_type": "PrimitiveFloat"},
        "29": {"class_type": "ComfyMathExpression"},
        "30": {"class_type": "LoadImage"}
      }
    }
  ]
}

In [ ]:
%%writefile /content/FRUTINOVELAS_kit/src/frutinovelas/__init__.py
﻿"""FRUTINOVELAS Studio."""

__version__ = "0.1.0"

In [ ]:
%%writefile /content/FRUTINOVELAS_kit/src/frutinovelas/services/__init__.py
"""Servicios de dominio compartidos por la API (y, más adelante, por el adaptador MCP)."""

In [ ]:
%%writefile /content/FRUTINOVELAS_kit/src/frutinovelas/services/voice_workflow.py
"""Reusable character voices through native H3 Ref2VA, without a TTS service.

This module intentionally uses only the standard library: the remote kit also
imports derive_voice_workflow to publish the same graph contract as the app.
Native node contract: ComfyUI 3541246a9c997f4b8c32f9f88f3c985d02021c79,
comfy_extras/nodes_minimax_h3.py and comfy_api/latest/_io.py::Autogrow.
Reference audio supplies timbre context; it is not placed on the output timeline.
"""

from __future__ import annotations

import copy
import json
import math
import re

VOICE_WORKFLOW = "H3_VOZ_PERSONAJES.api.json"
VOICE_UNET = "minimax_h3_ref2va_pruned_int8_convrot.safetensors"
VOICE_UNET_SHA256 = "9255f52b6677845ad238f20dfaafa94727053694127ab7f255c048f0f9365779"
START_GUIDE = "fruti_voice_start"
END_GUIDE = "fruti_voice_end"
MAX_VOICES = 3


class VoiceWorkflowError(ValueError):
    """A permanent, user-readable preparation failure before /prompt."""


def _node_inputs(graph, node_id, class_type):
    node = graph.get(node_id) if isinstance(graph, dict) else None
    if not isinstance(node, dict) or node.get("class_type") != class_type or not isinstance(node.get("inputs"), dict):
        raise VoiceWorkflowError(f"El flujo de voces no contiene {class_type} en el nodo {node_id}.")
    return node["inputs"]


def _shared_contract(graph):
    for node_id, class_type in {
        "1": "LoadImage", "5": "UNETLoader", "6": "CLIPLoader", "7": "VAELoader", "8": "VAELoader",
        "12": "RandomNoise", "14": "BasicScheduler", "15": "BasicGuider", "16": "SamplerCustomAdvanced",
        "18": "VAEDecodeAudio", "19": "CreateVideo", "20": "SaveVideo", "28": "PrimitiveFloat", "30": "LoadImage",
    }.items():
        _node_inputs(graph, node_id, class_type)
    if (
        graph["8"]["inputs"].get("vae_name") != "minimax_h3_audio_vae_fp32.safetensors"
        or graph["14"]["inputs"].get("steps") != 20
        or graph["19"]["inputs"].get("fps") != 24
        or graph["16"]["inputs"].get("latent_image") != ["9", 1]
        or graph["18"]["inputs"].get("vae") != ["8", 0]
        or graph["18"]["inputs"].get("samples") != ["16", 0]
        or graph["19"]["inputs"].get("audio") != ["18", 0]
    ):
        raise VoiceWorkflowError("El flujo de voces debe conservar el audio VAE, la salida nativa H3, 24 fps y 20 pasos.")


def derive_voice_workflow(base_graph, unet_name):
    """Return a private Ref2VA graph, with both visual anchors and no sample audio.

The kit passes its already overridden, validated start/end FL2VA graph. Encoder,
VAEs, sampler, canvas, duration calculation and output are preserved. Only the
diffusion weights, conditioning node and image-anchor connections change.
"""
    if unet_name != VOICE_UNET:
        raise VoiceWorkflowError("El perfil de voces requiere el checkpoint Ref2VA INT8 verificado.")
    _shared_contract(base_graph)
    inputs = _node_inputs(base_graph, "9", "MiniMaxH3ImageToVideo")
    if (
        (inputs.get("width"), inputs.get("height")) != (576, 1024)
        or inputs.get("clip") != ["6", 0]
        or inputs.get("vae") != ["7", 0]
        or "length" not in inputs
        or inputs.get("first_frame") != ["1", 0]
        or inputs.get("last_frame") != ["30", 0]
        or base_graph["15"]["inputs"].get("conditioning") != ["9", 0]
        or START_GUIDE in base_graph or END_GUIDE in base_graph
    ):
        raise VoiceWorkflowError("La variante de voces debe derivarse del flujo H3 original de inicio y final.")
    graph = copy.deepcopy(base_graph)
    graph["5"]["inputs"]["unet_name"] = unet_name
    graph["9"]["class_type"] = "MiniMaxH3ReferenceToVideo"
    graph["9"]["inputs"] = {
        key: copy.deepcopy(inputs[key]) for key in ("clip", "vae", "prompt", "width", "height", "length")
    }
    graph["9"]["inputs"].update({
        "audio_vae": ["8", 0], "ref_image_size": "match",
        "ref_images.ref_image_0": ["1", 0], "ref_images.ref_image_1": ["30", 0],
    })
    previous = ["9", 0]
    for node_id, image_id, frame in ((START_GUIDE, "1", 0), (END_GUIDE, "30", -1)):
        graph[node_id] = {
            "class_type": "MiniMaxH3AddGuide",
            "inputs": {"positive": previous, "latent": ["9", 1], "vae": ["7", 0], "image": [image_id, 0], "frame_idx": frame},
        }
        previous = [node_id, 0]
    graph["15"]["inputs"]["conditioning"] = previous
    validate_voice_workflow(graph)
    return graph


def validate_voice_workflow(graph):
    """Validate the published, unbound graph, not a graph with sample placeholders."""
    _shared_contract(graph)
    inputs = _node_inputs(graph, "9", "MiniMaxH3ReferenceToVideo")
    expected = {
        "clip": ["6", 0], "vae": ["7", 0], "audio_vae": ["8", 0],
        "width": 576, "height": 1024, "ref_image_size": "match",
        "ref_images.ref_image_0": ["1", 0], "ref_images.ref_image_1": ["30", 0],
    }
    if (
        graph["5"]["inputs"].get("unet_name") != VOICE_UNET
        or any(inputs.get(key) != value for key, value in expected.items())
        or "length" not in inputs or not isinstance(inputs.get("prompt"), str)
        or set(inputs) - (set(expected) | {"prompt", "length"})
        or graph["15"]["inputs"].get("conditioning") != [END_GUIDE, 0]
    ):
        raise VoiceWorkflowError("El flujo publicado no coincide con Ref2VA y sus referencias de voz e imagen.")
    previous = ["9", 0]
    for node_id, image_id, frame in ((START_GUIDE, "1", 0), (END_GUIDE, "30", -1)):
        guide = _node_inputs(graph, node_id, "MiniMaxH3AddGuide")
        if guide != {"positive": previous, "latent": ["9", 1], "vae": ["7", 0], "image": [image_id, 0], "frame_idx": frame}:
            raise VoiceWorkflowError("El flujo de voces no conserva las anclas visuales de inicio y final.")
        previous = [node_id, 0]
    if any(not isinstance(node, dict) or node.get("class_type") == "LoadAudio" for node in graph.values()):
        raise VoiceWorkflowError("El flujo publicado contiene un audio de ejemplo sin asignar.")


def validate_voice_references(references):
    if not isinstance(references, list) or len(references) > MAX_VOICES:
        raise VoiceWorkflowError("Cada escena admite hasta tres voces de personajes.")
    characters = set()
    total = 0.0
    for reference in references:
        if not isinstance(reference, dict):
            raise VoiceWorkflowError("La revisión contiene una referencia de voz no válida.")
        asset = reference.get("asset")
        character_id = reference.get("character_id")
        if (
            not isinstance(character_id, str) or not character_id or character_id in characters
            or not isinstance(reference.get("character_name"), str) or not reference["character_name"].strip()
            or not isinstance(reference.get("voice_description", ""), str)
            or not isinstance(reference.get("asset_id"), str)
            or not isinstance(asset, dict) or asset.get("id") != reference["asset_id"]
            or asset.get("kind") != "audio" or not isinstance(asset.get("sha256"), str)
            or not re.fullmatch(r"[0-9a-f]{64}", asset["sha256"])
        ):
            raise VoiceWorkflowError("Falta la muestra o la identidad de una voz guardada. Vuelve a guardar la selección.")
        duration = asset.get("duration_seconds")
        if type(duration) not in (int, float) or not math.isfinite(duration) or not 2 <= duration <= 15:
            raise VoiceWorkflowError("Cada muestra de voz para Ref2VA debe durar entre 2 y 15 segundos.")
        characters.add(character_id)
        total += duration
    if total > 15 + 1e-6:
        raise VoiceWorkflowError("Las muestras de las voces seleccionadas suman más de 15 segundos.")


def _speaker_label(prompt, name):
    """Reuse an explicit adjacent speaker ID; never guess or renumber one."""
    escaped = re.escape(name)
    labels = set(re.findall(rf"(?<!\w){escaped}\s*\(S(\d+)\)", prompt, re.IGNORECASE))
    labels.update(re.findall(rf"\(S(\d+)\)\s*{escaped}(?!\w)", prompt, re.IGNORECASE))
    return f" (S{next(iter(labels))})" if len(labels) == 1 else ""


def effective_voice_prompt(prompt, references, *, end_frame=False):
    """Compile native Ref2VA reference roles while preserving all literal dialogue.

The original prompt remains independently stored in the revision/job. Field
headings may change in this effective representation; dialogue is never rewritten.
"""
    validate_voice_references(references)
    if not references:
        return prompt
    directions = re.sub(r"<d>.*?</d>", "", prompt, flags=re.DOTALL)
    if re.search(r"<Audio\s+\d+>", directions, re.IGNORECASE):
        raise VoiceWorkflowError("Para usar las voces guardadas, identifica a los personajes por nombre; el programa asigna las etiquetas <Audio N>.")
    fields = {}
    pattern = r"(?m)^[ \t]*(subject_definitions|summary|retention_analysis|integrated_multimodal_description|detailed_description|overall_soundscape|non_diegetic_music)[ \t]*:"
    # Dialogue can contain newlines or words such as "summary:". Mask it at the
    # same offsets while locating headings, and slice only the original string.
    masked = re.sub(r"<d>.*?</d>", lambda match: re.sub(r"[^\n]", " ", match.group()), prompt, flags=re.DOTALL)
    matches = list(re.finditer(pattern, masked))
    prefix = prompt[:matches[0].start()] if matches else prompt
    for index, match in enumerate(matches):
        key = match.group(1)
        if key in fields:
            raise VoiceWorkflowError("El prompt repite una sección. Guarda una única descripción antes de usar voces.")
        fields[key] = prompt[match.end():matches[index + 1].start() if index + 1 < len(matches) else len(prompt)]
    if "integrated_multimodal_description" in fields and "detailed_description" in fields:
        raise VoiceWorkflowError("El prompt mezcla dos formatos de descripción. Guarda una única descripción de escena.")
    body = prefix + fields.get("detailed_description", fields.get("integrated_multimodal_description", ""))
    definitions = [fields.get("subject_definitions", "").strip(), "<Picture 1> is the supplied opening frame of the target scene."]
    retention = [fields.get("retention_analysis", "").strip(), "<Picture 1>: fully_preserved - preserve its opening composition and visible cast."]
    bindings = []
    if end_frame:
        definitions.append("<Picture 2> is the supplied final frame of the target scene.")
        retention.append("<Picture 2>: fully_preserved - use its composition at the end of the scene.")
    for index, reference in enumerate(references, 1):
        name = " ".join(reference["character_name"].split())
        character = "the character named " + json.dumps(name, ensure_ascii=False)
        speaker = character + _speaker_label(directions, name)
        definitions.append(f"<Audio {index}> is the voice-timbre reference for {speaker}.")
        retention.append(f"<Audio {index}>: reference - use its speaker timbre and vocal identity to deliver the new dialogue written for {character}.")
        line = f"Use the voice timbre from <Audio {index}> for the dialogue assigned to {speaker}."
        description = " ".join(reference.get("voice_description", "").split())
        if description:
            line += " Additional voice direction: " + description
        bindings.append(line)
    role_direction = (
        "Begin from <Picture 1>. " + ("End on <Picture 2>. " if end_frame else "")
        + "Use each reference only for the role assigned above. The spoken words are the new dialogue in the scene description. "
        "Keep the original language and existing speaker identifiers. The sample's original words and background sounds are not dialogue or ambience for this scene. "
        "If a speaking character is already visible, align their delivery and mouth movements with their speech. "
        "Otherwise keep that voice offscreen and preserve the visible cast. Visible listeners keep lips at rest."
    )
    summary = "[keyframe completion + audio reference] Generate the described scene with its supplied visual anchors and each character's referenced voice for the new dialogue."
    if fields.get("summary", "").strip():
        summary += "\n" + fields["summary"].strip()
    return "\n\n".join((
        "subject_definitions:\n" + "\n".join(item for item in definitions if item),
        "summary:\n" + summary,
        "retention_analysis:\n" + "\n".join(item for item in retention if item),
        "detailed_description:\n" + body.strip() + "\n\n" + "\n".join(bindings) + "\n" + role_direction,
        "overall_soundscape:\n" + fields.get("overall_soundscape", "Follow the ambience and physical sounds described in the scene.").strip(),
        "non_diegetic_music:\n" + fields.get("non_diegetic_music", "N/A.").strip(),
    ))


def _autogrow_accepts(inputs, group, prefix, kind, count):
    # /object_info exposes the V3 template, not its dotted live inputs. Accept a
    # future expanded representation too, but require all sockets we will use.
    if all(isinstance(inputs.get(f"{group}.{prefix}{i}"), list) and inputs[f"{group}.{prefix}{i}"][:1] == [kind] for i in range(count)):
        return True
    spec = inputs.get(group)
    if not isinstance(spec, list) or len(spec) < 2 or spec[0] != "COMFY_AUTOGROW_V3" or not isinstance(spec[1], dict):
        return False
    template = spec[1].get("template")
    if not isinstance(template, dict) or template.get("prefix") != prefix or template.get("min") != 0:
        return False
    maximum = template.get("max")
    nested = template.get("input")
    if type(maximum) is not int or maximum < count or not isinstance(nested, dict):
        return False
    required, optional = nested.get("required", {}), nested.get("optional", {})
    if not isinstance(required, dict) or not isinstance(optional, dict):
        return False
    values = list(required.values()) + list(optional.values())
    return len(values) == 1 and isinstance(values[0], list) and values[0][:1] == [kind]


def validate_voice_node_info(object_info):
    error = VoiceWorkflowError("ComfyUI no ofrece MiniMaxH3ReferenceToVideo con referencias de voz compatibles. Actualiza el kit remoto y comprueba la conexión.")
    node = object_info.get("MiniMaxH3ReferenceToVideo") if isinstance(object_info, dict) else None
    if not isinstance(node, dict) or node.get("output") != ["CONDITIONING", "LATENT"]:
        raise error
    inputs = node.get("input")
    if not isinstance(inputs, dict):
        raise error
    required, optional = inputs.get("required", {}), inputs.get("optional", {})
    if not isinstance(required, dict) or not isinstance(optional, dict):
        raise error
    fields = {"clip": "CLIP", "vae": "VAE", "audio_vae": "VAE", "prompt": "STRING", "width": "INT", "height": "INT", "length": "INT"}
    available = {**optional, **required}
    if set(required) - (set(fields) | {"ref_image_size"}):
        raise error
    for key, kind in fields.items():
        if not isinstance(available.get(key), list) or available[key][:1] != [kind]:
            raise error
    sizing = available.get("ref_image_size")
    options = None
    if isinstance(sizing, list) and sizing:
        if isinstance(sizing[0], list):
            options = sizing[0]
        elif sizing[0] == "COMBO" and len(sizing) > 1 and isinstance(sizing[1], dict):
            options = sizing[1].get("options")
    if not isinstance(options, list) or "match" not in options:
        raise error
    if not _autogrow_accepts(available, "ref_audios", "ref_audio_", "AUDIO", MAX_VOICES) or not _autogrow_accepts(available, "ref_images", "ref_image_", "IMAGE", 2):
        raise error


def prepare_voice_graph(graph, prompt, duration, seed, start, end, job_id, references, names):
    validate_voice_workflow(graph)
    validate_voice_references(references)
    if not references or len(names) != len(references) or any(not isinstance(name, str) or not name for name in names):
        raise VoiceWorkflowError("Falta subir una muestra de las voces seleccionadas.")
    result = copy.deepcopy(graph)
    result["1"]["inputs"]["image"] = start
    if end:
        result["30"]["inputs"]["image"] = end
    else:
        result["9"]["inputs"].pop("ref_images.ref_image_1")
        result.pop("30")
        result.pop(END_GUIDE)
        result["15"]["inputs"]["conditioning"] = [START_GUIDE, 0]
    for index, name in enumerate(names):
        node_id = f"fruti_voice_audio_{index}"
        if node_id in result:
            raise VoiceWorkflowError("El flujo usa un ID reservado para las muestras de voz.")
        result[node_id] = {"class_type": "LoadAudio", "inputs": {"audio": name}}
        result["9"]["inputs"][f"ref_audios.ref_audio_{index}"] = [node_id, 0]
    result["9"]["inputs"]["prompt"] = prompt
    result["12"]["inputs"]["noise_seed"] = seed
    result["28"]["inputs"]["value"] = duration
    result["20"]["inputs"]["filename_prefix"] = "FRUTINOVELAS/" + job_id
    return result

In [ ]:
# @title Tu flujo H3 (copia exacta; se verifica con SHA-256)
import hashlib

UI_SHA256 = "2153a2f6d289aaeb38882c4eb364a06ef3b039ad93addc4347b0473340c9e547"
UI_TEXT = r'''{
  "id": "h3-kungfu-fixed-r2v-576x1024",
  "revision": 6,
  "last_node_id": 30,
  "last_link_id": 57,
  "nodes": [
    {
      "id": 1,
      "type": "LoadImage",
      "pos": [
        30,
        120
      ],
      "size": [
        315,
        360
      ],
      "flags": {},
      "order": 0,
      "mode": 0,
      "inputs": [],
      "outputs": [
        {
          "name": "IMAGE",
          "type": "IMAGE",
          "links": [
            4
          ]
        },
        {
          "name": "MASK",
          "type": "MASK",
          "links": []
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0"
      },
      "widgets_values": [
        "tu_imagen_o_storyboard.png",
        "image"
      ],
      "widgets_values_named": {
        "image": "Storyboard01_Auto.png",
        "upload": "image"
      },
      "title": "IMAGEN INICIAL / STORYBOARD DE PRUEBA"
    },
    {
      "id": 5,
      "type": "UNETLoader",
      "pos": [
        430,
        85
      ],
      "size": [
        520,
        110
      ],
      "flags": {},
      "order": 1,
      "mode": 0,
      "inputs": [],
      "outputs": [
        {
          "name": "MODEL",
          "type": "MODEL",
          "links": [
            25,
            26
          ]
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0",
        "models": [
          {
            "name": "minimax_h3_ref2va_pruned_bf16.safetensors",
            "url": "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_ref2va_pruned_bf16.safetensors",
            "directory": "diffusion_models"
          }
        ]
      },
      "widgets_values": [
        "minimax_h3_fl2va_pruned_int8_convrot.safetensors",
        "default"
      ],
      "widgets_values_named": {
        "unet_name": "minimax_h3_ref2va_pruned_bf16.safetensors",
        "weight_dtype": "default"
      }
    },
    {
      "id": 6,
      "type": "CLIPLoader",
      "pos": [
        430,
        245
      ],
      "size": [
        520,
        115
      ],
      "flags": {},
      "order": 2,
      "mode": 0,
      "inputs": [],
      "outputs": [
        {
          "name": "CLIP",
          "type": "CLIP",
          "links": [
            1
          ]
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0",
        "models": [
          {
            "name": "qwen3vl_32b_minimax_h3_bf16.safetensors",
            "url": "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/e5eb578a89295337b8ff433a035929ce0279e0b6/text_encoders/qwen3vl_32b_minimax_h3_bf16.safetensors",
            "directory": "text_encoders"
          }
        ]
      },
      "widgets_values": [
        "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
        "minimax",
        "default"
      ],
      "widgets_values_named": {
        "clip_name": "qwen3vl_32b_minimax_h3_bf16.safetensors",
        "type": "minimax",
        "device": "default"
      }
    },
    {
      "id": 7,
      "type": "VAELoader",
      "pos": [
        430,
        415
      ],
      "size": [
        520,
        80
      ],
      "flags": {},
      "order": 3,
      "mode": 0,
      "inputs": [],
      "outputs": [
        {
          "name": "VAE",
          "type": "VAE",
          "links": [
            2,
            33
          ]
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0",
        "models": [
          {
            "name": "minimax_h3_video_vae_fp16.safetensors",
            "url": "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_fp16.safetensors",
            "directory": "vae"
          }
        ]
      },
      "widgets_values": [
        "minimax_h3_video_vae_int8_convrot.safetensors"
      ],
      "widgets_values_named": {
        "vae_name": "minimax_h3_video_vae_fp16.safetensors"
      }
    },
    {
      "id": 8,
      "type": "VAELoader",
      "pos": [
        430,
        530
      ],
      "size": [
        520,
        80
      ],
      "flags": {},
      "order": 4,
      "mode": 0,
      "inputs": [],
      "outputs": [
        {
          "name": "VAE",
          "type": "VAE",
          "links": [
            35
          ]
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0",
        "models": [
          {
            "name": "minimax_h3_audio_vae_fp32.safetensors",
            "url": "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors",
            "directory": "vae"
          }
        ]
      },
      "widgets_values": [
        "minimax_h3_audio_vae_fp32.safetensors"
      ],
      "widgets_values_named": {
        "vae_name": "minimax_h3_audio_vae_fp32.safetensors"
      }
    },
    {
      "id": 9,
      "type": "MiniMaxH3ImageToVideo",
      "pos": [
        1000,
        980
      ],
      "size": [
        590,
        470
      ],
      "flags": {},
      "order": 11,
      "mode": 0,
      "inputs": [
        {
          "name": "clip",
          "type": "CLIP",
          "link": 1
        },
        {
          "name": "vae",
          "type": "VAE",
          "link": 2
        },
        {
          "name": "prompt",
          "type": "STRING",
          "link": null,
          "widget": {
            "name": "prompt"
          }
        },
        {
          "name": "width",
          "type": "INT",
          "link": null,
          "widget": {
            "name": "width"
          }
        },
        {
          "name": "height",
          "type": "INT",
          "link": null,
          "widget": {
            "name": "height"
          }
        },
        {
          "name": "length",
          "type": "INT",
          "link": 53,
          "widget": {
            "name": "length"
          }
        },
        {
          "name": "first_frame",
          "type": "IMAGE",
          "link": 4,
          "shape": 7
        },
        {
          "name": "last_frame",
          "type": "IMAGE",
          "link": 57,
          "shape": 7
        }
      ],
      "outputs": [
        {
          "name": "positive",
          "type": "CONDITIONING",
          "links": [
            56
          ]
        },
        {
          "name": "LATENT",
          "type": "LATENT",
          "links": [
            31
          ]
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0"
      },
      "widgets_values": [
        "integrated_multimodal_description:\nAnimate the supplied image as a cinematic 10-second scene. Match the reference characters, clothing, lighting and environment. Use brisk ordinary movement and heightened television-drama acting with believable anatomy. Eyes react first, then eyebrows and head; show clear intention and natural blinks. Speaking characters use precise moderate mouth movement. Replace this example with the complete prompt for your current scene.\nIf the input is a storyboard, treat its panels as consecutive full-screen shots in reading order, not a persistent split-screen layout. Reconstruct one scene at a time and use direct camera cuts. The first frame is geometrically conditioned by the supplied image, so a collage may remain visible at the beginning; panel extraction is not guaranteed.\noverall_soundscape: Continuous environmental ambience underneath clear dialogue, with appropriate footsteps and object sounds.\nnon_diegetic_music: N/A.",
        576,
        1024,
        243
      ],
      "widgets_values_named": {
        "prompt": "integrated_multimodal_description:\n<Picture 1> is one storyboard sheet with four panels in a 2x2 grid. Read panel A at top left, B at top right, C at bottom left and D at bottom right. These panels define successive full-screen shots, not the output layout. Use each panel as a visual reference for its corresponding shot. The generated video fills the entire vertical frame with one scene at a time; the panel dividers and labels belong only to the reference sheet.\n[Shot 1] Cinematic 3D animation matching panel A. A complete charcoal-gray sedan travels left to right along a coastal city street at sunset. Track alongside at ordinary driving speed. Preserve the vehicle design, lighting and reference characters. Bruno, the avocado, drives from the left-hand seat; Darío, the green pear, sits in the passenger seat. Both wear seat belts. Animate grounded wheels, slight suspension movement and natural background motion.\n[Shot 2] At 00:02.000, the camera cuts directly to the passenger close-up in panel B. Darío looks toward the passing shop, recognizes it and turns his gaze slightly toward Bruno. His eyebrows stay relaxed with one small lift at recognition. Darío (S1), with a warm medium-low adult male voice, clear Latin American Spanish and brisk conversational delivery, says: <d>[Spanish] Tengo hambre. Ah, para ahí. Voy a comprar un helado.</d> Start promptly and finish before 6 seconds. Brief natural pause after the first sentence. Normal speaking volume, modest lip and jaw movement, steady head and attentive eyes. Preserve his exact reference clothing and fruit facial identity.\n[Shot 3] At 00:06.000, the camera cuts directly to Bruno's driver close-up in panel C. Bruno remains serious, relaxed and attentive to the road. Bruno (S2), with a rounded moderately deep adult male voice distinct from Darío, clear Latin American Spanish and firm concise delivery, says: <d>[Spanish] Para llevar.</d> Use slight dry humor in vocal timing rather than an exaggerated expression. His eyes briefly acknowledge Darío, then check the road and mirror. Hands remain naturally on the wheel. Finish before the next cut.\n[Shot 4] At 00:07.400, the camera cuts directly to the exterior parking view in panel D. The sedan is already moving slowly and aligned with the curb. Show the final short approach, smooth deceleration and small suspension settling as it stops beside the ice cream shop. Preserve left-to-right screen direction and the shop geometry. Both characters remain seated inside with doors closed. Darío looks toward the shop. End on the parked composition from panel D.\nUse ordinary real-time movement and brisk dialogue timing. Characters act immediately instead of holding long poses. Small facial amplitude retains clear intent. Maintain the same characters, outfits, car, seat positions and sunset throughout. Clean cuts introduce each new shot; full-screen scenes remain readable.\noverall_soundscape: Continuous soft engine, tire and street sounds remain audible across cuts and underneath both spoken turns. Tire noise diminishes when the car stops, leaving low engine idle and street ambience.\nnon_diegetic_music: N/A",
        "width": 576,
        "height": 1024,
        "length": 243,
        "ref_image_size": "max"
      },
      "title": "MiniMax H3 · IMAGE TO VIDEO · CAPTURA"
    },
    {
      "id": 12,
      "type": "RandomNoise",
      "pos": [
        1020,
        70
      ],
      "size": [
        340,
        95
      ],
      "flags": {},
      "order": 5,
      "mode": 0,
      "inputs": [],
      "outputs": [
        {
          "name": "NOISE",
          "type": "NOISE",
          "links": [
            27
          ]
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0"
      },
      "widgets_values": [
        42,
        "fixed"
      ],
      "widgets_values_named": {
        "noise_seed": 42,
        "control_after_generate": "fixed"
      }
    },
    {
      "id": 13,
      "type": "KSamplerSelect",
      "pos": [
        2140,
        60
      ],
      "size": [
        360,
        80
      ],
      "flags": {},
      "order": 6,
      "mode": 0,
      "inputs": [],
      "outputs": [
        {
          "name": "SAMPLER",
          "type": "SAMPLER",
          "links": [
            28
          ]
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0"
      },
      "widgets_values": [
        "res_multistep"
      ],
      "widgets_values_named": {
        "sampler_name": "res_multistep"
      }
    },
    {
      "id": 14,
      "type": "BasicScheduler",
      "pos": [
        2140,
        175
      ],
      "size": [
        360,
        140
      ],
      "flags": {},
      "order": 9,
      "mode": 0,
      "inputs": [
        {
          "name": "model",
          "type": "MODEL",
          "link": 25
        }
      ],
      "outputs": [
        {
          "name": "SIGMAS",
          "type": "SIGMAS",
          "links": [
            29
          ]
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0"
      },
      "widgets_values": [
        "simple",
        20,
        1
      ],
      "widgets_values_named": {
        "scheduler": "simple",
        "steps": 20,
        "denoise": 1
      }
    },
    {
      "id": 15,
      "type": "BasicGuider",
      "pos": [
        2180,
        470
      ],
      "size": [
        360,
        80
      ],
      "flags": {},
      "order": 12,
      "mode": 0,
      "inputs": [
        {
          "name": "model",
          "type": "MODEL",
          "link": 26
        },
        {
          "name": "conditioning",
          "type": "CONDITIONING",
          "link": 56
        }
      ],
      "outputs": [
        {
          "name": "GUIDER",
          "type": "GUIDER",
          "links": [
            30
          ]
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0"
      }
    },
    {
      "id": 16,
      "type": "SamplerCustomAdvanced",
      "pos": [
        2680,
        420
      ],
      "size": [
        385,
        160
      ],
      "flags": {},
      "order": 13,
      "mode": 0,
      "inputs": [
        {
          "name": "noise",
          "type": "NOISE",
          "link": 27
        },
        {
          "name": "guider",
          "type": "GUIDER",
          "link": 30
        },
        {
          "name": "sampler",
          "type": "SAMPLER",
          "link": 28
        },
        {
          "name": "sigmas",
          "type": "SIGMAS",
          "link": 29
        },
        {
          "name": "latent_image",
          "type": "LATENT",
          "link": 31
        }
      ],
      "outputs": [
        {
          "name": "output",
          "type": "LATENT",
          "links": [
            32,
            34
          ]
        },
        {
          "name": "denoised_output",
          "type": "LATENT",
          "links": []
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0"
      }
    },
    {
      "id": 17,
      "type": "VAEDecode",
      "pos": [
        3100,
        200
      ],
      "size": [
        315,
        80
      ],
      "flags": {},
      "order": 14,
      "mode": 0,
      "inputs": [
        {
          "name": "samples",
          "type": "LATENT",
          "link": 32
        },
        {
          "name": "vae",
          "type": "VAE",
          "link": 33
        }
      ],
      "outputs": [
        {
          "name": "IMAGE",
          "type": "IMAGE",
          "links": [
            36
          ]
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0"
      }
    },
    {
      "id": 18,
      "type": "VAEDecodeAudio",
      "pos": [
        3100,
        370
      ],
      "size": [
        315,
        80
      ],
      "flags": {},
      "order": 15,
      "mode": 0,
      "inputs": [
        {
          "name": "samples",
          "type": "LATENT",
          "link": 34
        },
        {
          "name": "vae",
          "type": "VAE",
          "link": 35
        }
      ],
      "outputs": [
        {
          "name": "AUDIO",
          "type": "AUDIO",
          "links": [
            37
          ]
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0"
      }
    },
    {
      "id": 19,
      "type": "CreateVideo",
      "pos": [
        3500,
        280
      ],
      "size": [
        320,
        105
      ],
      "flags": {},
      "order": 16,
      "mode": 0,
      "inputs": [
        {
          "name": "images",
          "type": "IMAGE",
          "link": 36
        },
        {
          "name": "audio",
          "type": "AUDIO",
          "link": 37,
          "shape": 7
        }
      ],
      "outputs": [
        {
          "name": "VIDEO",
          "type": "VIDEO",
          "links": [
            38
          ]
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0"
      },
      "widgets_values": [
        24,
        8
      ],
      "widgets_values_named": {
        "fps": 24,
        "bit_depth": 8
      }
    },
    {
      "id": 20,
      "type": "SaveVideo",
      "pos": [
        3900,
        280
      ],
      "size": [
        340,
        210
      ],
      "flags": {},
      "order": 17,
      "mode": 0,
      "inputs": [
        {
          "name": "video",
          "type": "VIDEO",
          "link": 38
        }
      ],
      "outputs": [
        {
          "name": "video",
          "type": "VIDEO",
          "links": []
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0"
      },
      "widgets_values": [
        "video/FRUTINOVELAS_I2V_CAPTURA",
        "auto",
        "auto"
      ],
      "widgets_values_named": {
        "filename_prefix": "video/FRUTINOVELAS_STORYBOARD01_BF16",
        "format": "auto",
        "codec": "auto"
      }
    },
    {
      "id": 27,
      "type": "Note",
      "title": "LEEME | EQUILIBRADO 9:16",
      "pos": [
        1000,
        70
      ],
      "size": [
        600,
        650
      ],
      "flags": {},
      "order": 7,
      "mode": 0,
      "inputs": [],
      "outputs": [],
      "properties": {},
      "widgets_values": [
        "Perfil equilibrado: FL2VA INT8 convrot, encoder NVFP4 AWQ, video VAE INT8 convrot y audio FP32 originales. 20 pasos base, sin LoRA Turbo, res_multistep/simple, denoise 1. Canvas 576x1024: 9:16 exacto y resolucion reducida respecto al canvas nativo de H3. Validar calidad en la GPU real; no se ha ejecutado inferencia. Se conserva la duracion original de 10 s, calculada a 243 frames / 24 fps (~10.125 s). Cambiala en DURACION y adapta el prompt a esa duracion. Carga tu imagen inicial en LoadImage. El JSON original contiene widgets_values_named con un prompt distinto al widget visible: se conservan ambos sin reescribir el dialogo; al exportar API comprobar el prompt que usa la interfaz. Para el formato limpio usa una sola imagen, no un collage. Audio y guardado originales. Variante inicio/final: selecciona ambas imagenes antes de ejecutar. El nuevo LoadImage alimenta last_frame; no es un generador de imagenes. Para una sola referencia utiliza la variante INICIO."
      ]
    },
    {
      "id": 28,
      "type": "PrimitiveFloat",
      "pos": [
        950,
        60
      ],
      "size": [
        270,
        70
      ],
      "flags": {},
      "order": 8,
      "mode": 0,
      "inputs": [],
      "outputs": [
        {
          "name": "FLOAT",
          "type": "FLOAT",
          "links": [
            52
          ]
        }
      ],
      "title": "DURACION | segundos (10 por defecto)",
      "properties": {
        "Node name for S&R": "PrimitiveFloat",
        "cnr_id": "comfy-core",
        "ver": "0.33.0"
      },
      "widgets_values": [
        10
      ],
      "color": "#322",
      "bgcolor": "#533"
    },
    {
      "id": 29,
      "type": "ComfyMathExpression",
      "pos": [
        1280,
        60
      ],
      "size": [
        225,
        0
      ],
      "flags": {},
      "order": 10,
      "mode": 0,
      "inputs": [
        {
          "label": "a",
          "name": "values.a",
          "type": "FLOAT,INT,BOOLEAN",
          "link": 52
        },
        {
          "name": "values.b",
          "shape": 7,
          "type": "FLOAT,INT,BOOLEAN",
          "link": null
        }
      ],
      "outputs": [
        {
          "name": "FLOAT",
          "type": "FLOAT",
          "links": []
        },
        {
          "name": "INT",
          "type": "INT",
          "links": [
            53
          ]
        },
        {
          "name": "BOOL",
          "type": "BOOLEAN",
          "links": []
        }
      ],
      "properties": {
        "Node name for S&R": "ComfyMathExpression",
        "cnr_id": "comfy-core",
        "ver": "0.33.0"
      },
      "widgets_values": [
        "max(5, round(a * 24)) + (5 - (max(5, round(a * 24)) % 17)) % 17"
      ],
      "title": "FOTOGRAMAS | rejilla H3 automatica"
    },
    {
      "id": 30,
      "type": "LoadImage",
      "pos": [
        30,
        560
      ],
      "size": [
        315,
        360
      ],
      "flags": {},
      "order": 1,
      "mode": 0,
      "inputs": [],
      "outputs": [
        {
          "name": "IMAGE",
          "type": "IMAGE",
          "links": [
            57
          ]
        },
        {
          "name": "MASK",
          "type": "MASK",
          "links": []
        }
      ],
      "properties": {
        "cnr_id": "comfy-core",
        "ver": "0.33.0"
      },
      "widgets_values": [
        "tu_imagen_final.png",
        "image"
      ],
      "widgets_values_named": {
        "image": "tu_imagen_final.png",
        "upload": "image"
      },
      "title": "IMAGEN FINAL / FINAL FRAME"
    }
  ],
  "links": [
    [
      1,
      6,
      0,
      9,
      0,
      "CLIP"
    ],
    [
      2,
      7,
      0,
      9,
      1,
      "VAE"
    ],
    [
      4,
      1,
      0,
      9,
      6,
      "IMAGE"
    ],
    [
      25,
      5,
      0,
      14,
      0,
      "MODEL"
    ],
    [
      26,
      5,
      0,
      15,
      0,
      "MODEL"
    ],
    [
      27,
      12,
      0,
      16,
      0,
      "NOISE"
    ],
    [
      28,
      13,
      0,
      16,
      2,
      "SAMPLER"
    ],
    [
      29,
      14,
      0,
      16,
      3,
      "SIGMAS"
    ],
    [
      30,
      15,
      0,
      16,
      1,
      "GUIDER"
    ],
    [
      31,
      9,
      1,
      16,
      4,
      "LATENT"
    ],
    [
      32,
      16,
      0,
      17,
      0,
      "LATENT"
    ],
    [
      33,
      7,
      0,
      17,
      1,
      "VAE"
    ],
    [
      34,
      16,
      0,
      18,
      0,
      "LATENT"
    ],
    [
      35,
      8,
      0,
      18,
      1,
      "VAE"
    ],
    [
      36,
      17,
      0,
      19,
      0,
      "IMAGE"
    ],
    [
      37,
      18,
      0,
      19,
      1,
      "AUDIO"
    ],
    [
      38,
      19,
      0,
      20,
      0,
      "VIDEO"
    ],
    [
      52,
      28,
      0,
      29,
      0,
      "FLOAT"
    ],
    [
      53,
      29,
      1,
      9,
      5,
      "INT"
    ],
    [
      56,
      9,
      0,
      15,
      1,
      "CONDITIONING"
    ],
    [
      57,
      30,
      0,
      9,
      7,
      "IMAGE"
    ]
  ],
  "groups": [],
  "config": {},
  "extra": {
    "ds": {
      "scale": 0.7,
      "offset": [
        10,
        20
      ]
    },
    "notes": "Ajuste del JSON del usuario: FL2VA I2V, 20 pasos, 576x1024 (9:16), 24 fps. Pesos, sampler, seed, audio, duracion y conexiones originales conservados. JSON validado estructuralmente; sin inferencia GPU."
  },
  "version": 0.4
}
'''
datos = UI_TEXT.encode("utf-8")
if hashlib.sha256(datos).hexdigest() != UI_SHA256:
    raise SystemExit("❌ El flujo incrustado no coincide con el original (SHA-256).")
UI_PATH.write_bytes(datos)
print("✅ Flujo H3 escrito sin cambios:", UI_PATH, "·", UI_SHA256[:12])

## 2 · Diagnóstico

In [ ]:
# @title 2 · Diagnóstico del entorno
import platform
import shutil
import subprocess

MANIFEST, PROFILE, vw = cargar_kit()


def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


gpu_line = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader,nounits")
if not gpu_line:
    raise SystemExit("❌ No hay GPU. Colab: Entorno de ejecución → Cambiar tipo → GPU. RunPod: crea el pod con GPU.")
gpu_name, vram_mb, compute_cap, driver = [x.strip() for x in gpu_line.splitlines()[0].split(",")]
vram_gb = float(vram_mb) / 1024
ram_gb = int(open("/proc/meminfo").read().split("MemTotal:")[1].split()[0]) / 1024 / 1024
disk = shutil.disk_usage(MODELS_DIR)

try:
    import torch

    torch_info = f"{torch.__version__} · CUDA {torch.version.cuda} · disponible={torch.cuda.is_available()}"
except Exception as exc:
    torch_info = f"no disponible ({exc})"


def componentes_activos():
    vistos = []
    for perfil in PERFILES_ACTIVOS:
        for comp in MANIFEST["profiles"][perfil]["components"]:
            if comp not in vistos:
                vistos.append(comp)
    return vistos


def ruta_local(comp_key):
    c = MANIFEST["components"][comp_key]
    return MODELS_DIR / "hf" / c["repo_id"].replace("/", "__") / c["path"]


pendiente = sum(MANIFEST["components"][k]["size"] for k in componentes_activos()
                if not (ruta_local(k).exists() and ruta_local(k).stat().st_size == MANIFEST["components"][k]["size"]))

print(f"GPU:     {gpu_name} · {vram_gb:.1f} GB VRAM · compute {compute_cap} · driver {driver}")
print(f"RAM:     {ram_gb:.1f} GB")
print(f"Disco:   {disk.free / 1e9:.1f} GB libres en {MODELS_DIR}")
print(f"Python:  {platform.python_version()}")
print(f"PyTorch: {torch_info}")
print(f"FFmpeg:  {'sí' if shutil.which('ffmpeg') else 'no'}")
print(f"Descarga pendiente: {pendiente / 1e9:.1f} GB")

avisos = []
if vram_gb < 40:
    avisos.append("VRAM < 40 GB: H3 puede fallar por memoria o ir muy lento (requisito no verificado).")
if ram_gb < 50:
    avisos.append("RAM < 50 GB: activa 'RAM alta' si tu plan lo permite; la carga de modelos puede fallar.")
if disk.free < pendiente + 15e9:
    avisos.append("Disco insuficiente para los modelos seleccionados (+15 GB de margen).")
try:
    if float(compute_cap) < 10.0:
        avisos.append("GPU anterior a Blackwell: el encoder NVFP4 AWQ puede ir más lento o no ser compatible (no verificado).")
except ValueError:
    pass
for a in avisos:
    print("⚠️", a)
estado(diagnostico={"gpu": gpu_name, "vram_gb": round(vram_gb, 1), "compute_cap": compute_cap,
                    "ram_gb": round(ram_gb, 1), "disco_libre_gb": round(disk.free / 1e9, 1), "torch": torch_info})
print("✅ Diagnóstico terminado" + (" con avisos" if avisos else ""))

## 3 · Instalación

In [ ]:
# @title 3 · Instalar ComfyUI (versión fijada) y dependencias
import subprocess


def run(cmd, cwd=None):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    if r.returncode != 0:
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
        raise RuntimeError(f"Falló: {cmd}")
    return r.stdout


if not (COMFY_DIR / ".git").is_dir():
    COMFY_DIR.mkdir(parents=True, exist_ok=True)
    run(f"git init -q {COMFY_DIR}")
    run(f"git -C {COMFY_DIR} remote add origin {COMFY_REPO}")
actual = subprocess.run(f"git -C {COMFY_DIR} rev-parse HEAD", shell=True, capture_output=True, text=True).stdout.strip()
if actual != COMFY_COMMIT:
    run(f"git -C {COMFY_DIR} fetch -q --depth 1 origin {COMFY_COMMIT}")
    run(f"git -C {COMFY_DIR} checkout -q --force {COMFY_COMMIT}")
print("ComfyUI en commit", run(f"git -C {COMFY_DIR} rev-parse HEAD").strip(), f"({COMFY_TAG})")

import shutil

if not PIP_MARKER.exists() or PIP_MARKER.read_text().strip() != COMFY_COMMIT:
    run(f"{sys.executable} -m pip install -q -r requirements.txt", cwd=COMFY_DIR)
    run(f'{sys.executable} -m pip install -q -U "huggingface_hub[hf_xet]" playwright pillow requests aiohttp')
    run(f"{sys.executable} -m playwright install --with-deps chromium")
    PIP_MARKER.write_text(COMFY_COMMIT)
else:
    print("Dependencias ya instaladas para este commit.")
if not shutil.which("ffprobe"):
    run("apt-get update -qq && apt-get install -y -qq ffmpeg")

import importlib.metadata as md

for paquete in ["torch", "comfyui-frontend-package", "comfy-kitchen", "aiohttp", "huggingface_hub", "playwright"]:
    try:
        print(f"  {paquete} {md.version(paquete)}")
    except md.PackageNotFoundError:
        print(f"  {paquete} (no instalado)")
estado(comfyui_commit=COMFY_COMMIT)
print("✅ Instalación lista (no se instalan SageAttention ni xformers)")

## 4–5 · Modelos

In [ ]:
# @title 4 · Plan de modelos (manifiesto con revisiones fijadas)
MANIFEST, PROFILE, vw = cargar_kit()
print(f"{'componente':40} {'GB':>7}  estado")
total = 0
for k in componentes_activos():
    c = MANIFEST["components"][k]
    p = ruta_local(k)
    if p.exists() and p.stat().st_size == c["size"]:
        st = "presente"
    elif p.exists():
        st = "incompleto"
    else:
        st = "por descargar"
        total += c["size"]
    print(f"{Path(c['path']).name[:40]:40} {c['size'] / 1e9:7.2f}  {st}")
libre = shutil.disk_usage(MODELS_DIR).free
print(f"\nPor descargar: {total / 1e9:.1f} GB · libre en destino: {libre / 1e9:.1f} GB")
if libre < total + 5e9:
    raise SystemExit("❌ No hay espacio suficiente. Desactiva perfiles opcionales o usa un entorno con más disco.")
print("✅ Hay espacio suficiente")

In [ ]:
# @title 5 · Descargar y verificar modelos
import hashlib

os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
from huggingface_hub import hf_hub_download

HF_TOKEN = secreto("HF_TOKEN")


def sha256_archivo(path, size):
    h = hashlib.sha256()
    leido, siguiente = 0, 0.1
    with open(path, "rb") as f:
        while bloque := f.read(16 * 1024 * 1024):
            h.update(bloque)
            leido += len(bloque)
            if leido / size >= siguiente:
                print(f"    sha256 {leido / size:4.0%}", end="\r")
                siguiente += 0.1
    print(" " * 30, end="\r")
    return h.hexdigest()


def verificar(c, path):
    st = path.stat()
    if st.st_size != c["size"]:
        return False, f"tamaño {st.st_size} ≠ {c['size']}"
    if not VERIFICAR_SHA256:
        return True, "tamaño correcto (SHA-256 desactivado)"
    marca = path.with_name(path.name + ".sha256ok")
    clave = f"{st.st_size}:{int(st.st_mtime)}:{c['sha256']}"
    if marca.exists() and marca.read_text() == clave:
        return True, "SHA-256 correcto (ya verificado)"
    digest = sha256_archivo(path, c["size"])
    if digest != c["sha256"]:
        return False, f"SHA-256 {digest} ≠ {c['sha256']}"
    marca.write_text(clave)
    return True, "SHA-256 correcto"


for k in componentes_activos():
    c = MANIFEST["components"][k]
    rev = MANIFEST["repos"][c["repo_id"]]
    path = ruta_local(k)
    nombre = Path(c["path"]).name
    print(f"• {nombre} ({c['size'] / 1e9:.2f} GB)")
    if path.exists():
        ok, motivo = verificar(c, path)
        if not ok:
            print("    ⚠️", motivo, "→ se vuelve a descargar")
            path.unlink()
    if not path.exists():
        t0 = time.time()
        hf_hub_download(repo_id=c["repo_id"], filename=c["path"], revision=rev,
                        local_dir=MODELS_DIR / "hf" / c["repo_id"].replace("/", "__"), token=HF_TOKEN)
        print(f"    descargado en {time.time() - t0:.0f} s")
        ok, motivo = verificar(c, path)
        if not ok:
            path.unlink(missing_ok=True)
            raise RuntimeError(f"{nombre}: {motivo}")
    print("    ✅", motivo)
    enlace = COMFY_DIR / "models" / c["dest"] / nombre
    enlace.parent.mkdir(parents=True, exist_ok=True)
    if enlace.is_symlink() and enlace.resolve() != path.resolve():
        enlace.unlink()
    if not enlace.exists() and not enlace.is_symlink():
        enlace.symlink_to(path)
estado(modelos={k: MANIFEST["components"][k]["sha256"] for k in componentes_activos()})
print("✅ Modelos listos y enlazados en", COMFY_DIR / "models")

## 6 · Flujos

In [ ]:
# @title 6 · Preparar flujos (tu H3 + plantillas Qwen opcionales)
import requests

ui = json.loads(UI_PATH.read_text(encoding="utf-8"))
print("Flujo H3:", UI_PATH.name, "·", vw.detect_format(ui), "·", len(ui["nodes"]), "nodos")
if vw.detect_format(ui) != "ui":
    raise SystemExit("❌ El flujo H3 no tiene formato de interfaz.")

TEMPLATES = {}
for perfil in PERFILES_ACTIVOS:
    plantilla = MANIFEST["profiles"][perfil].get("template")
    if not plantilla:
        continue
    destino = KIT_DIR / "workflows" / "templates" / Path(plantilla).name
    if not destino.exists():
        url = f"https://raw.githubusercontent.com/Comfy-Org/workflow_templates/{TEMPLATES_COMMIT}/{plantilla}"
        r = requests.get(url, timeout=60, headers={"User-Agent": UA})
        r.raise_for_status()
        destino.write_bytes(r.content)
    datos = json.loads(destino.read_text(encoding="utf-8"))
    if vw.detect_format(datos) != "ui":
        raise SystemExit(f"❌ {destino.name} no es un flujo de interfaz.")
    TEMPLATES[perfil] = destino
    print("Plantilla:", destino.name, "·", len(datos["nodes"]), "nodos")
print("✅ Flujos preparados")

## 7 · ComfyUI

In [ ]:
# @title 7 · Arrancar ComfyUI y comprobar nodos y modelos
import subprocess

import requests

LOCAL = f"http://127.0.0.1:{COMFY_PORT}"


def comfy_vivo():
    try:
        return requests.get(LOCAL + "/system_stats", timeout=3).ok
    except Exception:
        return False


def cola_log(path, n=60):
    if path.exists():
        print("\n".join(path.read_text(errors="replace").splitlines()[-n:]))


if not comfy_vivo():
    log = open(LOG_DIR / "comfyui.log", "ab")
    p = subprocess.Popen(
        [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT), "--disable-auto-launch",
         "--output-directory", str(DATA_DIR / "comfy_output"), "--input-directory", str(DATA_DIR / "comfy_input")],
        cwd=COMFY_DIR, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    PROCS["comfyui"] = p.pid
    guardar_pids()
    t0 = time.time()
    while not comfy_vivo():
        if p.poll() is not None:
            cola_log(LOG_DIR / "comfyui.log")
            raise RuntimeError("ComfyUI se cerró al arrancar (log arriba).")
        if time.time() - t0 > 900:
            raise RuntimeError("ComfyUI no respondió en 15 minutos.")
        time.sleep(3)
    print(f"ComfyUI arrancado en {time.time() - t0:.0f} s")
else:
    print("ComfyUI ya estaba en marcha")

STATS = requests.get(LOCAL + "/system_stats", timeout=30).json()
OBJECT_INFO = requests.get(LOCAL + "/object_info", timeout=300).json()
(DATA_DIR / "object_info.json").write_text(json.dumps(OBJECT_INFO))
print("Versión ComfyUI:", STATS.get("system", {}).get("comfyui_version"))

faltan = sorted({n["type"] for n in ui["nodes"] if n["type"] not in ("Note", "MarkdownNote")} - set(OBJECT_INFO))
if faltan:
    raise SystemExit(f"❌ Nodos ausentes en este ComfyUI: {faltan}")
print("✅ Todos los nodos del flujo H3 existen")

LOADER_FIELD = {"diffusion_models": ("UNETLoader", "unet_name"), "text_encoders": ("CLIPLoader", "clip_name"),
                "vae": ("VAELoader", "vae_name"), "loras": ("LoraLoaderModelOnly", "lora_name")}
sin_ver = []
for k in componentes_activos():
    c = MANIFEST["components"][k]
    clase, campo = LOADER_FIELD[c["dest"]]
    spec = (OBJECT_INFO.get(clase, {}).get("input", {}).get("required", {}) or {}).get(campo)
    opciones = vw.combo_options(spec) or []
    if Path(c["path"]).name not in opciones:
        sin_ver.append(f"{clase}.{campo}: {Path(c['path']).name}")
if sin_ver:
    raise SystemExit("❌ ComfyUI no ve estos modelos: " + "; ".join(sin_ver))
print("✅ ComfyUI ve todos los modelos seleccionados")

In [ ]:
# @title 7B · Exportar a formato API (automático) y validar
import hashlib
import subprocess

MANIFEST, PROFILE, vw = cargar_kit()


def exportar(ui_path, out_path):
    r = subprocess.run([sys.executable, str(KIT_DIR / "scripts" / "export_workflow_api.py"), "--comfy", LOCAL,
                        "--ui", str(ui_path), "--out", str(out_path)], capture_output=True, text=True)
    print(r.stdout.strip())
    if r.returncode != 0:
        print(r.stderr[-4000:])
        raise RuntimeError(f"No se pudo exportar {ui_path.name}")
    return json.loads(out_path.read_text(encoding="utf-8"))


def sha_json(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


INFORME = {"generado": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "comfyui_commit": COMFY_COMMIT, "flujos": {}}
crudo = exportar(UI_PATH, DATA_DIR / "workflows" / "H3_export_crudo.api.json")
omitidos = vw.ui_nodes_missing_in_api(ui, crudo)
print("Nodos de la interfaz que no van a la API (las notas son normales):", omitidos or "ninguno")
print("Nodo 28 (duración):", crudo.get("28", {}).get("inputs"))
print("Nodo 29 (fórmula):", {k: v for k, v in crudo.get("29", {}).get("inputs", {}).items() if k != "expression"})

overrides = MANIFEST["profiles"][PERFIL_VIDEO]["loader_overrides"]
variantes = {"fl2va": crudo, "i2va": vw.derive_i2va(crudo, PROFILE)}
errores_totales = []
for modo, grafo in variantes.items():
    r1 = vw.check_profile(grafo, PROFILE, modo)
    grafo_final, cambios = vw.apply_loader_overrides(grafo, overrides)
    r2 = vw.validate_api_graph(grafo_final, OBJECT_INFO)
    errores = r1["errors"] + r2["errors"]
    avisos = r1["warnings"] + r2["warnings"]
    nombre = PROFILE["workflow_api_files"][modo]
    for a in avisos:
        print(f"  [{modo}] aviso:", a)
    for e in errores:
        print(f"  [{modo}] ERROR:", e)
    for cbio in cambios:
        print(f"  [{modo}] perfil {PERFIL_VIDEO}:", cbio)
    if not errores:
        destino = API_DIR / nombre
        destino.write_text(json.dumps(grafo_final, ensure_ascii=False, indent=2), encoding="utf-8")
        INFORME["flujos"][nombre] = {"modo": modo, "nodos": len(grafo_final), "sha256": sha_json(destino),
                                     "cambios_perfil": cambios, "avisos": avisos}
        print(f"✅ {nombre} ({len(grafo_final)} nodos)")
    errores_totales += errores

if ACTIVAR_VOCES_PERSONAJES:
    from frutinovelas.services.voice_workflow import VOICE_WORKFLOW, derive_voice_workflow, validate_voice_node_info

    # Mantener ambos extremos visuales en el flujo publicado. Studio conecta las voces reales.
    base_voz, cambios = vw.apply_loader_overrides(crudo, overrides)
    unet = Path(MANIFEST["components"]["h3_diffusion_ref2va_pruned_int8"]["path"]).name
    try:
        validate_voice_node_info(OBJECT_INFO)
        voz = derive_voice_workflow(base_voz, unet)
        rv = vw.validate_api_graph(voz, OBJECT_INFO)
    except ValueError as exc:
        rv = {"errors": [str(exc)], "warnings": []}
    for a in rv["warnings"]:
        print("  [voces] aviso:", a)
    for e in rv["errors"]:
        print("  [voces] ERROR:", e)
    if not rv["errors"]:
        destino = API_DIR / VOICE_WORKFLOW
        destino.write_text(json.dumps(voz, ensure_ascii=False, indent=2), encoding="utf-8")
        INFORME["flujos"][VOICE_WORKFLOW] = {
            "modo": "ref2va", "perfil": "h3_voice_references", "nodos": len(voz),
            "sha256": sha_json(destino), "cambios_perfil": cambios, "avisos": rv["warnings"],
        }
        print(f"✅ {VOICE_WORKFLOW} ({len(voz)} nodos; muestras por personaje desde Studio)")
    errores_totales += rv["errors"]

NOMBRES_QWEN = {"qwen2512_t2i": "QWEN_2512_T2I.api.json", "qwen_edit2511_fp8mixed": "QWEN_EDIT_2511.api.json",
                "qwen_edit2511_bf16": "QWEN_EDIT_2511.api.json"}
for perfil, plantilla in TEMPLATES.items():
    api_q = exportar(plantilla, DATA_DIR / "workflows" / (plantilla.stem + ".api.json"))
    api_q, cambios = vw.apply_loader_overrides(api_q, MANIFEST["profiles"][perfil]["loader_overrides"])
    rq = vw.validate_api_graph(api_q, OBJECT_INFO)
    for a in rq["warnings"]:
        print(f"  [{perfil}] aviso:", a)
    for e in rq["errors"]:
        print(f"  [{perfil}] ERROR:", e)
    if not rq["errors"]:
        destino = API_DIR / NOMBRES_QWEN[perfil]
        destino.write_text(json.dumps(api_q, ensure_ascii=False, indent=2), encoding="utf-8")
        INFORME["flujos"][destino.name] = {"perfil": perfil, "nodos": len(api_q), "sha256": sha_json(destino),
                                           "cambios_perfil": cambios, "avisos": rq["warnings"]}
        print(f"✅ {destino.name} ({len(api_q)} nodos)")
    errores_totales += rq["errors"]

if "qwen21_bf16" in PERFILES_ACTIVOS:
    from frutinovelas.services.qwen21_workflow import WORKFLOWS, build_graph

    api_q = build_graph()
    rq = vw.validate_api_graph(api_q, OBJECT_INFO)
    errores_totales += rq["errors"]
    for error in rq["errors"]:
        print("  [qwen21_bf16] ERROR:", error)
    if not rq["errors"]:
        for filename in WORKFLOWS:
            destino = API_DIR / filename
            destino.write_text(json.dumps(api_q, ensure_ascii=False, indent=2), encoding="utf-8")
            INFORME["flujos"][filename] = {"perfil": "qwen21_bf16", "nodos": len(api_q),
                "sha256": sha_json(destino), "cambios_perfil": [], "avisos": rq["warnings"]}
            print(f"✅ {filename} (BF16, 1152×2048, 40 pasos)")

(DATA_DIR / "informe_exportacion.json").write_text(json.dumps(INFORME, ensure_ascii=False, indent=2))
if errores_totales:
    raise SystemExit(f"❌ {len(errores_totales)} error(es) de validación (detalle arriba). No se publican esos flujos.")
print("✅ Flujos API exportados y validados contra este ComfyUI:", ", ".join(INFORME["flujos"]))

## 8 · Conexión segura

In [ ]:
# @title 8 · Gateway con token + acceso público (túnel Cloudflare o proxy RunPod)
import re
import secrets as _secrets
import subprocess
import urllib.request

import requests

TOKEN_FILE = DATA_DIR / "secrets" / "gateway_token.txt"
TOKEN_PERSIST = PERSIST_DIR / "secrets" / "gateway_token.txt" if PERSIST_DIR else None
token, origen = secreto("FRUTI_GATEWAY_TOKEN"), "secreto FRUTI_GATEWAY_TOKEN"
if not token and TOKEN_FILE.exists():
    token, origen = TOKEN_FILE.read_text().strip(), "guardado en esta máquina"
if not token and TOKEN_PERSIST and TOKEN_PERSIST.exists():
    token, origen = TOKEN_PERSIST.read_text().strip(), f"guardado en {PERSIST_DIR}"
if not token:
    token, origen = _secrets.token_urlsafe(32), "generado ahora"
if len(token) < 16:
    raise SystemExit("❌ El token del gateway debe tener al menos 16 caracteres.")
TOKEN_FILE.write_text(token)
os.chmod(TOKEN_FILE, 0o600)
if TOKEN_PERSIST and origen == "generado ahora":
    TOKEN_PERSIST.parent.mkdir(parents=True, exist_ok=True)
    TOKEN_PERSIST.write_text(token)
    os.chmod(TOKEN_PERSIST, 0o600)
GATEWAY_TOKEN = token
AUTH = {"Authorization": f"Bearer {GATEWAY_TOKEN}", "User-Agent": UA}
print("Token del gateway:", origen, "· termina en …" + GATEWAY_TOKEN[-4:])


def escribir_identidad(extra=None):
    ident = {
        "servicio": f"frutinovelas-comfyui-{PLATAFORMA}",
        "plataforma": PLATAFORMA,
        "notebook_version": NOTEBOOK_VERSION,
        "comfyui_version": STATS.get("system", {}).get("comfyui_version"),
        "comfyui_commit": COMFY_COMMIT,
        "python": STATS.get("system", {}).get("python_version"),
        "pytorch": STATS.get("system", {}).get("pytorch_version"),
        "gpu": [{"nombre": d.get("name"), "vram_total": d.get("vram_total")} for d in STATS.get("devices", [])],
        "perfil_video": PERFIL_VIDEO,
        "perfiles": PERFILES_ACTIVOS,
        "modelos": {k: {"archivo": Path(MANIFEST["components"][k]["path"]).name, "sha256": MANIFEST["components"][k]["sha256"]}
                    for k in componentes_activos()},
        "workflows": {n: d["sha256"] for n, d in INFORME["flujos"].items()},
        "workflow_ui_sha256": PROFILE["workflow_ui_sha256"],
        "estado_compatibilidad": "pendiente_prueba_real",
        "actualizado": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    anterior = DATA_DIR / "identity.json"
    if anterior.exists():
        viejo = json.loads(anterior.read_text())
        misma_configuracion = all(viejo.get(k) == ident.get(k) for k in
                                  ("comfyui_commit", "perfiles", "modelos", "workflows", "workflow_ui_sha256"))
        if misma_configuracion:
            ident["pruebas"] = viejo.get("pruebas", {})
            ident["estado_compatibilidad"] = viejo.get("estado_compatibilidad", ident["estado_compatibilidad"])
    ident.update(extra or {})
    anterior.write_text(json.dumps(ident, ensure_ascii=False, indent=2))
    return ident


IDENTIDAD = escribir_identidad()
GW = f"http://127.0.0.1:{GATEWAY_PORT}"


def gw_vivo():
    try:
        return requests.get(GW + "/healthz", timeout=3).ok
    except Exception:
        return False


# Con el proxy de RunPod el gateway debe escuchar en todas las interfaces; sigue exigiendo token.
# ComfyUI (8188) se queda siempre en 127.0.0.1 y nunca se expone.
GATEWAY_LISTEN = "0.0.0.0" if MODO_TUNEL == "runpod_proxy" else "127.0.0.1"
if gw_vivo() and estado().get("gateway_listen") != GATEWAY_LISTEN and PROCS.get("gateway"):
    try:
        os.killpg(int(PROCS.pop("gateway")), signal.SIGTERM)
    except ProcessLookupError:
        pass
    time.sleep(1)
if not gw_vivo():
    env = dict(os.environ, FRUTI_GATEWAY_TOKEN_FILE=str(TOKEN_FILE))
    env.pop("FRUTI_GATEWAY_TOKEN", None)
    p = subprocess.Popen([sys.executable, str(KIT_DIR / "scripts" / "comfy_gateway.py"), "--listen", GATEWAY_LISTEN,
                          "--port", str(GATEWAY_PORT), "--upstream", LOCAL, "--workflows-dir", str(API_DIR),
                          "--identity-file", str(DATA_DIR / "identity.json")],
                         env=env, stdout=open(LOG_DIR / "gateway.log", "ab"), stderr=subprocess.STDOUT, start_new_session=True)
    PROCS["gateway"] = p.pid
    guardar_pids()
    estado(gateway_listen=GATEWAY_LISTEN)
    for _ in range(40):
        if gw_vivo():
            break
        if p.poll() is not None:
            cola_log(LOG_DIR / "gateway.log")
            raise RuntimeError("El gateway se cerró (log arriba).")
        time.sleep(0.5)
assert requests.get(GW + "/system_stats", timeout=10).status_code == 401, "El gateway debería rechazar peticiones sin token"
assert requests.get(GW + "/system_stats", headers=AUTH, timeout=10).ok, "El gateway no reenvía a ComfyUI"
print("✅ Gateway local OK (sin token → 401, con token → 200)")

CF_BIN = KIT_DIR / "bin" / "cloudflared"
PUBLIC_URL = None
if MODO_TUNEL == "runpod_proxy":
    pod = os.environ.get("RUNPOD_POD_ID")
    if not pod:
        raise SystemExit("❌ RUNPOD_POD_ID no está definido: el modo runpod_proxy solo funciona dentro de un pod de RunPod.")
    PUBLIC_URL = f"https://{pod}-{GATEWAY_PORT}.proxy.runpod.net"
elif MODO_TUNEL in ("quick", "named"):
    if not CF_BIN.exists():
        url = f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64"
        urllib.request.urlretrieve(url, CF_BIN)
        os.chmod(CF_BIN, 0o755)
    print(subprocess.run([str(CF_BIN), "--version"], capture_output=True, text=True).stdout.strip())
    previo = estado().get("public_url")
    cf_pid = PROCS.get("cloudflared")
    if cf_pid and vivo(cf_pid) and previo:
        PUBLIC_URL = previo
        print("Túnel ya activo")
    elif MODO_TUNEL == "quick":
        cf_log = LOG_DIR / "cloudflared.log"
        cf_log.write_text("")
        p = subprocess.Popen([str(CF_BIN), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{GATEWAY_PORT}"],
                             stdout=open(cf_log, "ab"), stderr=subprocess.STDOUT, start_new_session=True)
        PROCS["cloudflared"] = p.pid
        guardar_pids()
        for _ in range(90):
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", cf_log.read_text(errors="replace"))
            if m:
                PUBLIC_URL = m.group(0)
                break
            time.sleep(1)
        if not PUBLIC_URL:
            cola_log(cf_log)
            raise RuntimeError("No se obtuvo la URL del Quick Tunnel (log arriba).")
    else:
        cf_token, host = secreto("CF_TUNNEL_TOKEN"), secreto("CF_TUNNEL_HOSTNAME")
        if not (cf_token and host):
            raise SystemExit("❌ Para MODO_TUNEL='named' define CF_TUNNEL_TOKEN y CF_TUNNEL_HOSTNAME.")
        env = dict(os.environ, TUNNEL_TOKEN=cf_token)
        p = subprocess.Popen([str(CF_BIN), "tunnel", "--no-autoupdate", "run"], env=env,
                             stdout=open(LOG_DIR / "cloudflared.log", "ab"), stderr=subprocess.STDOUT, start_new_session=True)
        PROCS["cloudflared"] = p.pid
        guardar_pids()
        PUBLIC_URL = "https://" + host.removeprefix("https://").rstrip("/")

if PUBLIC_URL:
    publico_ok = False
    for _ in range(60):
        try:
            if requests.get(PUBLIC_URL + "/healthz", timeout=10, headers={"User-Agent": UA}).ok:
                publico_ok = True
                break
        except Exception:
            pass
        time.sleep(2)
    if not publico_ok:
        if MODO_TUNEL == "runpod_proxy":
            raise RuntimeError(f"{PUBLIC_URL} no responde. En RunPod: Edit Pod → Expose HTTP Ports → añade {GATEWAY_PORT} "
                               "(el pod se reinicia; vuelve a ejecutar desde la celda 1).")
        raise RuntimeError(f"{PUBLIC_URL} no responde. Revisa {LOG_DIR / 'cloudflared.log'}")
    r = requests.get(PUBLIC_URL + "/system_stats", headers=AUTH, timeout=30)
    if not r.ok:
        raise RuntimeError(f"El acceso público responde {r.status_code} con token.")
    if requests.get(PUBLIC_URL + "/system_stats", headers={"User-Agent": UA}, timeout=30).status_code != 401:
        raise RuntimeError("⚠️ El acceso público NO exige token. Detén el pod y revisa la configuración.")
    print("✅ Acceso público verificado (sin token → 401, con token → 200)")
estado(public_url=PUBLIC_URL)
print("URL pública:", PUBLIC_URL or "(sin acceso público: solo accesible dentro de la máquina)")

## 9 · Datos para el programa

In [ ]:
# @title 9 · Datos de conexión para FRUTINOVELAS Studio (sin secretos)
DESCARGAR_ARCHIVO = False  # @param {type:"boolean"}
conexion = {
    "formato": "frutinovelas.conexion_comfyui",
    "version": 1,
    "comfyui_url": PUBLIC_URL,
    "autenticacion": "Authorization: Bearer <token>",
    "token": "NO INCLUIDO: cópialo desde la celda 9B y pégalo en el programa",
    "plataforma": PLATAFORMA,
    "tunel": MODO_TUNEL,
    "flujos": sorted(INFORME["flujos"]),
    "identidad": {k: IDENTIDAD[k] for k in ("comfyui_version", "comfyui_commit", "perfil_video", "workflow_ui_sha256")},
    "creado": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "aviso": {"quick": "La URL de un Quick Tunnel cambia en cada sesión.",
              "runpod_proxy": "La URL depende del ID del pod: se mantiene mientras uses el mismo pod."}.get(MODO_TUNEL, ""),
}
ruta_conexion = DATA_DIR / "conexion_frutinovelas.json"
ruta_conexion.write_text(json.dumps(conexion, ensure_ascii=False, indent=2))
if PERSIST_DIR:
    (PERSIST_DIR / "conexion_frutinovelas.json").write_text(ruta_conexion.read_text())
print("═" * 70)
print("  URL de ComfyUI para el programa:")
print("   ", PUBLIC_URL)
print("  Flujos publicados:", ", ".join(conexion["flujos"]))
print("═" * 70)
print("Archivo:", ruta_conexion)
if DESCARGAR_ARCHIVO and IN_COLAB:
    from google.colab import files

    files.download(str(ruta_conexion))

In [ ]:
# @title 9B · Token del gateway (paso privado)
MOSTRAR_TOKEN = False  # @param {type:"boolean"}
if MOSTRAR_TOKEN:
    print(GATEWAY_TOKEN)
    print("\n⚠️ Cópialo al programa y después borra la salida de esta celda (⋮ → Borrar salida).")
else:
    print("Marca MOSTRAR_TOKEN y vuelve a ejecutar para ver el token. Termina en …" + GATEWAY_TOKEN[-4:])

## Conectar FRUTINOVELAS Studio (en tu PC)

1. Actualiza el código de **Studio y MCP** a la misma versión que esta libreta.
   Abre `iniciar.cmd` en tu PC y reinicia el proceso MCP si estaba abierto.
2. Copia la **URL** de la celda 9 y el **token** de la celda 9B. En Studio abre
   **Conexión ComfyUI**, pega ambos y pulsa **Guardar y comprobar**. Studio verifica
   la identidad del gateway y el hash del flujo; no hace falta importar JSON a mano.
3. Para un diagnóstico opcional desde PowerShell, dentro de `frutinovelas-studio`:

   ```powershell
   $env:FRUTI_COMFY_TOKEN = "PEGA_AQUI_EL_TOKEN"
   uv run python scripts/check_colab_connection.py --url https://XXXX.trycloudflare.com
   ```

   Descarga los flujos API exportados a `workflows/api/` y verifica sus SHA-256.
4. Si activaste voces, la exportación de la celda 7B debe incluir
   `H3_VOZ_PERSONAJES.api.json`. Guarda una muestra por personaje en la biblioteca,
   selecciona sus voces en **Audio y voces**, escribe los diálogos nuevos y guarda
   la escena. Se puede usar una imagen inicial o también una final.
5. La IA puede leer `docs/ESTILOS_VISUALES.md` por el recurso MCP
   `frutinovelas://guides/visual-styles`; `get_directing_guidance` lo incorpora para
   imagen, animación y revisión. **Leerlo no selecciona el estilo**: úsalo solo por
   petición expresa. La búsqueda en internet corresponde a la IA conectada, no a
   este servidor de inferencia.
6. Si Colab se desconecta: vuelve a ejecutar las celdas 1–9 (el token guardado en Drive se reutiliza; la URL del Quick Tunnel cambia).

Guardar prompts, estilos y muestras no inicia una inferencia. Al generar el primer
clip, revisa la imagen, los diálogos y la voz; las pruebas automáticas y la exportación
del grafo no miden la calidad generada.


## 10 · Pruebas reales (opcional)

In [ ]:
# @title 10 · Pruebas reales opcionales (usan GPU y tiempo)
PROBAR_H3_SOLO_INICIO = False  # @param {type:"boolean"}
PROBAR_H3_INICIO_FINAL = False  # @param {type:"boolean"}
PROBAR_QWEN = False  # @param {type:"boolean"}
PROMPT_PRUEBA = "Una fresa sonriente camina por un mercado colorido al atardecer, cámara estable, luz cálida."  # @param {type:"string"}
SEGUNDOS = 5  # @param {type:"slider", min:4, max:15, step:1}
SEMILLA = 42  # @param {type:"integer"}

import subprocess
import uuid

import requests
from IPython.display import Image as IPImage
from IPython.display import Video, display
from PIL import Image, ImageDraw

MANIFEST, PROFILE, vw = cargar_kit()


def imagen_prueba(path, c1, c2, texto):
    w, h = 576, 1024
    img = Image.new("RGB", (w, h))
    d = ImageDraw.Draw(img)
    for y in range(h):
        t = y / (h - 1)
        d.line([(0, y), (w, y)], fill=tuple(int(a + (b - a) * t) for a, b in zip(c1, c2)))
    d.ellipse([w // 2 - 120, h // 2 - 120, w // 2 + 120, h // 2 + 120], fill=(230, 40, 70))
    d.text((40, 60), texto, fill=(255, 255, 255))
    img.save(path)
    return path


def subir(path):
    with open(path, "rb") as f:
        r = requests.post(GW + "/upload/image", headers=AUTH, files={"image": (path.name, f, "image/png")},
                          data={"type": "input", "overwrite": "true"}, timeout=120)
    r.raise_for_status()
    j = r.json()
    return f"{j['subfolder']}/{j['name']}" if j.get("subfolder") else j["name"]


def encolar(grafo):
    pid = str(uuid.uuid4())
    r = requests.post(GW + "/prompt", headers=AUTH, timeout=120,
                      json={"prompt": grafo, "prompt_id": pid, "client_id": "frutinovelas-colab"})
    if r.status_code != 200 or r.json().get("node_errors"):
        print(r.text[:4000])
        raise RuntimeError("ComfyUI rechazó el trabajo")
    return pid


def esperar(pid, limite_s=3 * 3600):
    t0 = time.time()
    while time.time() - t0 < limite_s:
        h = requests.get(f"{GW}/history/{pid}", headers=AUTH, timeout=60).json()
        if pid in h:
            entrada = h[pid]
            st = entrada.get("status", {})
            if st.get("completed") or st.get("status_str") in ("success", "error"):
                print(f"  terminado en {time.time() - t0:.0f} s · {st.get('status_str')}")
                if st.get("status_str") != "success":
                    for msg in st.get("messages", []):
                        print("   ", msg)
                    raise RuntimeError("El trabajo terminó con error")
                return entrada
        print(f"  generando… {time.time() - t0:.0f} s", end="\r")
        time.sleep(10)
    raise TimeoutError("La prueba superó el tiempo límite")


def descargar_salidas(entrada):
    rutas = []
    for salida in entrada.get("outputs", {}).values():
        for valor in salida.values():
            if not isinstance(valor, list):
                continue
            for item in valor:
                if isinstance(item, dict) and item.get("filename"):
                    r = requests.get(GW + "/view", headers=AUTH, timeout=600, params={
                        "filename": item["filename"], "subfolder": item.get("subfolder", ""), "type": item.get("type", "output")})
                    r.raise_for_status()
                    destino = DATA_DIR / "tests" / item["filename"]
                    destino.write_bytes(r.content)
                    rutas.append(destino)
    return rutas


def ffprobe(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-count_frames", "-show_entries",
                        "stream=codec_type,width,height,r_frame_rate,nb_read_frames:format=duration", "-of", "json", str(path)],
                       capture_output=True, text=True)
    return json.loads(r.stdout or "{}")


def registrar(nombre, resultado):
    ident = json.loads((DATA_DIR / "identity.json").read_text())
    ident.setdefault("pruebas", {})[nombre] = resultado
    if all(v.get("ok") for v in ident["pruebas"].values()):
        ident["estado_compatibilidad"] = "prueba_real_ok"
    (DATA_DIR / "identity.json").write_text(json.dumps(ident, ensure_ascii=False, indent=2))


def prueba_h3(modo):
    frames, seg = vw.quantize_frames(SEGUNDOS, PROFILE["duration_rule"])
    print(f"▶ H3 {modo}: {SEGUNDOS} s pedidos → {frames} frames ({seg:.3f} s reales)")
    base = json.loads((API_DIR / PROFILE["workflow_api_files"][modo]).read_text())
    valores = {"start_image": subir(imagen_prueba(DATA_DIR / "tests" / "inicio.png", (250, 170, 60), (120, 40, 90), "INICIO")),
               "positive_prompt": PROMPT_PRUEBA, "seed": int(SEMILLA), "duration": float(SEGUNDOS),
               "filename_prefix": f"video/FRUTINOVELAS_PRUEBA_{modo.upper()}"}
    if modo == "fl2va":
        valores["end_image"] = subir(imagen_prueba(DATA_DIR / "tests" / "final.png", (60, 40, 140), (20, 160, 150), "FINAL"))
    t0 = time.time()
    salidas = descargar_salidas(esperar(encolar(vw.materialize(base, PROFILE, valores))))
    videos = [p for p in salidas if p.suffix.lower() in (".mp4", ".webm", ".mov")]
    if not videos:
        raise RuntimeError(f"No se encontró vídeo en las salidas: {salidas}")
    info = ffprobe(videos[0])
    v = next((s for s in info.get("streams", []) if s.get("codec_type") == "video"), {})
    audio = any(s.get("codec_type") == "audio" for s in info.get("streams", []))
    print(f"  {videos[0].name}: {v.get('width')}x{v.get('height')} · {v.get('nb_read_frames')} frames · "
          f"{float(info.get('format', {}).get('duration', 0)):.2f} s · audio={'sí' if audio else 'no'}")
    ok = str(v.get("nb_read_frames")) == str(frames)
    if not ok:
        print(f"  ⚠️ Frames esperados {frames}")
    registrar(f"h3_{modo}", {"ok": ok, "segundos": SEGUNDOS, "frames": v.get("nb_read_frames"),
                             "tiempo_s": round(time.time() - t0), "fecha": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})
    display(Video(str(videos[0]), embed=True, width=288))


if PROBAR_H3_SOLO_INICIO:
    prueba_h3("i2va")
if PROBAR_H3_INICIO_FINAL:
    prueba_h3("fl2va")
if PROBAR_QWEN:
    ruta_q = API_DIR / ("QWEN_21_T2I.api.json" if ACTIVAR_QWEN_21_BF16 else "QWEN_2512_T2I.api.json")
    if not ruta_q.exists():
        raise SystemExit("Activa ACTIVAR_QWEN_21_BF16 en la celda 1 y repite las celdas 4–8.")
    print("▶ Prueba", ruta_q.name)
    t0 = time.time()
    imgs = descargar_salidas(esperar(encolar(json.loads(ruta_q.read_text()))))
    registrar("qwen21_bf16" if ACTIVAR_QWEN_21_BF16 else "qwen2512_t2i", {"ok": bool(imgs), "tiempo_s": round(time.time() - t0)})
    for p in imgs:
        display(IPImage(filename=str(p), width=320))
if not (PROBAR_H3_SOLO_INICIO or PROBAR_H3_INICIO_FINAL or PROBAR_QWEN):
    print("Ninguna prueba marcada. Recomendado: PROBAR_H3_SOLO_INICIO con 5 s.")

## 11 · Cierre

In [ ]:
# @title 11 · Guardar copia de la sesión y (opcional) detener procesos
DETENER_PROCESOS = False  # @param {type:"boolean"}
import shutil

if PERSIST_DIR:
    sesion = PERSIST_DIR / "sesiones" / time.strftime("%Y%m%d_%H%M%S")
    (sesion / "workflows_api").mkdir(parents=True, exist_ok=True)
    for nombre in ["identity.json", "conexion_frutinovelas.json", "informe_exportacion.json", "estado.json"]:
        if (DATA_DIR / nombre).exists():
            shutil.copy2(DATA_DIR / nombre, sesion / nombre)
    for p in API_DIR.glob("*.api.json"):
        shutil.copy2(p, sesion / "workflows_api" / p.name)
    shutil.copytree(LOG_DIR, sesion / "logs", dirs_exist_ok=True)
    if any((DATA_DIR / "tests").iterdir()):
        shutil.copytree(DATA_DIR / "tests", sesion / "pruebas", dirs_exist_ok=True)
    print("✅ Guardado en", sesion, "(sin secretos)")
else:
    print("Sin almacenamiento persistente: no se guarda nada fuera del entorno.")

if DETENER_PROCESOS:
    for nombre in ["cloudflared", "gateway", "comfyui"]:
        pid = PROCS.get(nombre)
        if pid and vivo(pid):
            try:
                os.killpg(int(pid), signal.SIGTERM)
            except ProcessLookupError:
                pass
            for _ in range(20):
                if not vivo(pid):
                    break
                time.sleep(0.5)
            print(f"Detenido: {nombre}")
        PROCS.pop(nombre, None)
    guardar_pids()
    estado(public_url=None)
    print("✅ Procesos detenidos. Colab: desconecta el entorno. RunPod: detén el pod (el volumen sigue cobrando aunque esté parado).")
else:
    print("Los procesos siguen en marcha. Marca DETENER_PROCESOS para pararlos.")